In [ ]:
import os
import sys

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")


import numpy as np
from util_manager import PathManager, get_network
import tvm
from tvm import auto_scheduler

TARGET = tvm.target.Target("cuda")

def get_tasks(mod, params, path_manager, verbose=True, get_pkl=True):
    if get_pkl:
        tasks, task_weights = path_manager.tasks_pkl_use()
    
    if get_pkl is False or tasks is None:
        print("Extract tasks...")
        tasks, task_weights = auto_scheduler.extract_tasks(mod["main"], params, TARGET)
        if path_manager.tasks_pkl_check() is False:
            path_manager.tasks_pkl_save(tasks, task_weights)

    if verbose:
        for idx, task in enumerate(tasks):
            print("========== Task %d : %s  (workload key: %s) ==========" % (idx, task.desc, task.workload_key))
            print(task.compute_dag)
    
    print(f"Total tasks length : {len(tasks)}")
    return tasks, task_weights


In [ ]:
from types import SimpleNamespace

args = SimpleNamespace(
    network="resnet_50",
    batch_size=1,
    dtype="float32",
    layout="NHWC",
    timenow=None,
    json=None
)

mod, params, input_shape, output_shape = get_network(args.network, args.batch_size, args.layout, dtype=args.dtype)
path_manager = PathManager(args.network, input_shape, args, None, None)
tasks, task_weights = get_tasks(None, params, path_manager, verbose=False, get_pkl=True)
tasks, task_weights = zip(*sorted(zip(tasks, task_weights), key=lambda x: x[0].desc))

search_policies = []
for idx, (task, weight) in enumerate(zip(tasks, task_weights)):
    print(f"T{idx} : {task.desc} ({weight})")
    search_policies.append(
        auto_scheduler.SketchPolicy(task, auto_scheduler.XGBModel())
    )

Getting network resnet_50...

Using json : /root/work/tvm-ansor/gallery/logs_json/resnet_50/(1,224,224,3)-0307_1556.json
Load tasks from /root/work/tvm-ansor/gallery/ansor_tasks_pkl/resnet_50-(1,224,224,3).pkl
Total tasks length : 29
T0 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu (3)
T1 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu_1 (4)
T2 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu_2 (6)
T3 : vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu_3 (3)
T4 : vm_mod_fused_nn_conv2d (1)
T5 : vm_mod_fused_nn_conv2d_1 (1)
T6 : vm_mod_fused_nn_conv2d_2 (1)
T7 : vm_mod_fused_nn_conv2d_3 (1)
T8 : vm_mod_fused_nn_conv2d_add (2)
T9 : vm_mod_fused_nn_conv2d_add_1 (3)
T10 : vm_mod_fused_nn_conv2d_add_2 (5)
T11 : vm_mod_fused_nn_conv2d_add_3 (2)
T12 : vm_mod_fused_nn_conv2d_add_add_nn_relu (1)
T13 : vm_mod_fused_nn_conv2d_add_add_nn_relu_1 (1)
T14 : vm_mod_fused_nn_conv2d_a

In [32]:
from common import load_and_register_network
tasks = load_and_register_network("/root/work/tvm-ansor/gallery/dataset/network_info/((llama_tiny,[(1,64)]),cuda).task.pkl")

In [3]:
builtins_min = min

"""
SymbolicState: task.compute_dag의 stage/iter 구조를 그대로 복사하고,
transform step 적용 시 split/unroll factor를 symbolic variable로 표현하는 객체.

C++ loop_state.cc의 State/Stage/Iterator 구조를 Python으로 미러링합니다.

변수 네이밍:
  - SplitStep:             sp_{step_idx}_{length_idx}
  - FollowSplitStep:       src SplitStep의 sym_name 재사용
  - FollowFusedSplitStep:  src SplitStep들의 sym_name 곱
  - PragmaStep (unroll):   ur_{step_idx}

ComputeAt extent 복원 전략 (lazy):
  ComputeAt는 extent를 None으로만 설정.
  이후 Fuse/Split/FFSP에서 None extent를 만나면 그 시점(step_idx)에서
  ReplayStepsPartial + InferBound를 호출하여 concrete extent를 동적 복원.
  이렇게 해야 부모 stage의 split 결과가 반영된 정확한 extent를 얻을 수 있음.
"""
from collections import OrderedDict
import math

# ── Annotation 문자열 매핑 (C++ IteratorAnnotationString 동일) ──
ANNOTATION_STR = {
    0: "for",           # kNone
    1: "unroll",        # kUnroll
    2: "vectorize",     # kVectorize
    3: "parallel",      # kParallel
    4: "vthread",       # kVThread
    5: "blockIdx.x",    # kBlockX
    6: "threadIdx.x",   # kThreadX
    7: "blockIdx.y",    # kBlockY
    8: "threadIdx.y",   # kThreadY
    9: "blockIdx.z",    # kBlockZ
    10: "threadIdx.z",  # kThreadZ
    11: "tensorize",    # kTensorize
}

# ── ComputeAtKind ──
CA_ROOT    = 0  # kRoot
CA_INLINED = 1  # kInlined
CA_ITER    = 2  # kIter


class SymExpr:
    """
    Symbolic expression wrapper.
    실제 값(int)이면 그냥 int, symbolic이면 문자열.
    연산(ceil div, mul 등)을 문자열로 합성.
    """
    def __init__(self, val):
        self.val = val

    @property
    def is_concrete(self):
        return isinstance(self.val, int)

    def __repr__(self):
        return str(self.val) if self.val is not None else "None"

    def __str__(self):
        return str(self.val) if self.val is not None else "None"

    def __int__(self):
        if self.is_concrete:
            return self.val
        raise ValueError(f"Cannot convert symbolic '{self.val}' to int")

    @staticmethod
    def ceildiv(a, b):
        if isinstance(a, SymExpr): a = a.val
        if isinstance(b, SymExpr): b = b.val
        if isinstance(a, int) and isinstance(b, int):
            return SymExpr((a + b - 1) // b)
        return SymExpr(f"ceil({a}/({b}))")

    @staticmethod
    def _needs_parens_for_mul(s):
        """문자열 expression이 mul에서 사용될 때 괄호가 필요한지 판단.
        최외곽 레벨에서 + 또는 - 연산자가 있으면 True."""
        if not isinstance(s, str):
            return False
        depth = 0
        i = 0
        while i < len(s):
            c = s[i]
            if c == '(':
                depth += 1
            elif c == ')':
                depth -= 1
            elif depth == 0 and c == '+':
                return True
            elif depth == 0 and c == '-' and i > 0 and s[i-1] == ' ':
                return True
            i += 1
        return False

    @staticmethod
    def mul(a, b):
        if isinstance(a, SymExpr): a = a.val
        if isinstance(b, SymExpr): b = b.val
        if isinstance(a, int) and isinstance(b, int):
            return SymExpr(a * b)
        if a == 1: return SymExpr(b)
        if b == 1: return SymExpr(a)
        # 연산자 우선순위 보호: 최외곽에 +/- 포함된 표현식은 괄호 추가
        a_str = str(a)
        b_str = str(b)
        if SymExpr._needs_parens_for_mul(a_str):
            a_str = f"({a_str})"
        if SymExpr._needs_parens_for_mul(b_str):
            b_str = f"({b_str})"
        return SymExpr(f"{a_str}*{b_str}")

    @staticmethod
    def product(items):
        result = SymExpr(1)
        for item in items:
            result = SymExpr.mul(result, item)
        return result

    @staticmethod
    def min(a, b):
        """min(a, b) — both concrete → int min, otherwise symbolic min string."""
        if a is None or b is None:
            return a if b is None else b
        if isinstance(a, SymExpr): a_val = a.val
        else: a_val = a
        if isinstance(b, SymExpr): b_val = b.val
        else: b_val = b
        if isinstance(a_val, int) and isinstance(b_val, int):
            return SymExpr(builtins_min(a_val, b_val))
        return SymExpr(f"min({a_val},{b_val})")


class SymIter:
    """Iterator (C++ Iterator 대응)"""
    def __init__(self, name, extent, annotation=0, iter_kind=0):
        self.name = name
        self.extent = extent         # SymExpr or None
        self.annotation = annotation # int
        self.iter_kind = iter_kind   # int

    def clone(self):
        return SymIter(self.name,
                       SymExpr(self.extent.val) if self.extent else None,
                       self.annotation, self.iter_kind)

    def __repr__(self):
        ann = ANNOTATION_STR.get(self.annotation, "?")
        if self.extent is not None:
            return f"{ann} {self.name} (0,{self.extent})"
        else:
            return f"{ann} {self.name} (None)"


class SymStage:
    """Stage (C++ Stage 대응)"""
    def __init__(self, op_name, op_type, iters, compute_at=CA_ROOT,
                 auto_unroll_max_step=None, storage_offset=0, dtype="float32"):
        self.op_name = op_name
        self.op_type = op_type
        self.iters = list(iters)
        self.compute_at = compute_at
        self.auto_unroll_max_step = auto_unroll_max_step
        self.storage_offset = storage_offset
        self.attach_stage_id = None
        self.attach_iter_id = None
        self.dtype = dtype  # e.g. "float32", "float16", "int8"

    @property
    def dtype_bytes(self):
        """dtype의 바이트 수를 반환."""
        import tvm
        return tvm.DataType(self.dtype).bits // 8


def eval_sym_extent(expr, sym_map):
    """SymExpr의 문자열을 sym_map으로 치환하여 eval로 계산"""
    if expr is None:
        return None
    s_val = str(expr)
    if s_val == "None":
        return None
    try:
        return int(s_val)
    except ValueError:
        pass
    evaluated = s_val
    evaluated = evaluated.replace("ceil(", "math.ceil(")
    for sym_name in sorted(sym_map.keys(), key=len, reverse=True):
        if sym_map[sym_name] is not None:
            evaluated = evaluated.replace(sym_name, str(sym_map[sym_name]))
    try:
        return int(eval(evaluated))
    except Exception:
        return f"EVAL_FAIL({s_val}→{evaluated})"


In [17]:
# ═══════════════════════════════════════════════════════════════
# SymbolicState  – 순수 상태 (stages, sym_map, 출력, extent 조회)
# TransformApplier – transform step 적용 로직
# SymParamManager – 파라미터 조회 / 검증 / 랜덤 생성
# ═══════════════════════════════════════════════════════════════


# ─────────────────────────────────────────────────────────────
#  SymbolicState: 순수 상태 객체
# ─────────────────────────────────────────────────────────────
class SymbolicState:
    """
    Symbolic 버전의 auto_scheduler State.
    stages, sym_map, 내부 메타데이터를 보유하는 순수 상태 객체.
    transform step 적용은 TransformApplier, 파라미터 관리는 SymParamManager가 담당.
    """

    @staticmethod
    def _safe_int_extent(extent_expr):
        """TIR extent를 int로 변환. Sub/Add 등 심볼릭이면 simplify 후 재시도."""
        if extent_expr is None:
            return None
        try:
            return int(extent_expr)
        except TypeError:
            import tvm
            simplified = tvm.arith.Analyzer().simplify(extent_expr)
            return int(simplified)

    def __init__(self, compute_dag):
        self.stages = []
        self.sym_map = OrderedDict()
        self.compute_dag = compute_dag
        self._state = None  # TransformApplier.apply_steps에서 설정
        self._ca_saved_extents = {}  # {(stage_id, iter_id): SymExpr}
        self._split_sym_products = {}  # {(stage_id, step_idx): SymExpr}
        self._cache_read_consumer = {}  # {cache_read_stage_id: consumer_stage_id}
        self._cache_read_stencil_info = {}  # {cr_stage_id: {cr_axis_idx: (stride, sp_order, rd_order)}}
        self._shared_fused_extents = {}  # {stage_id: SymExpr}

        for sid, op in enumerate(compute_dag.ops):
            if hasattr(op, 'axis'):
                dtype = str(op.output(0).dtype) if hasattr(op, 'output') else "float32"
                iters = []
                for axis in op.axis:
                    name = str(axis.var.name)
                    ext = self._safe_int_extent(axis.dom.extent) if axis.dom is not None else None
                    iters.append(SymIter(name, SymExpr(ext) if ext is not None else None,
                                         annotation=0, iter_kind=0))
                for axis in op.reduce_axis:
                    name = str(axis.var.name)
                    ext = self._safe_int_extent(axis.dom.extent) if axis.dom is not None else None
                    iters.append(SymIter(name, SymExpr(ext) if ext is not None else None,
                                         annotation=0, iter_kind=1))
                self.stages.append(SymStage(op.name, 'compute', iters, dtype=dtype))
            else:
                dtype = str(op.output(0).dtype) if hasattr(op, 'output') else "float32"
                self.stages.append(SymStage(op.name, 'placeholder', [], dtype=dtype))

    # ─── 내부 데이터 shift (CacheRead/CacheWrite stage 삽입 시) ───
    def _shift_ca_saved_extents(self, inserted_stage_id):
        """CacheRead/CacheWrite로 stage가 삽입될 때, saved extents와 관련 데이터 key 업데이트."""
        new_saved = {}
        for (sid, iid), expr in self._ca_saved_extents.items():
            new_sid = sid + 1 if sid >= inserted_stage_id else sid
            new_saved[(new_sid, iid)] = expr
        self._ca_saved_extents = new_saved

        new_split_prods = {}
        for (sid, step_idx), expr in self._split_sym_products.items():
            new_sid = sid + 1 if sid >= inserted_stage_id else sid
            new_split_prods[(new_sid, step_idx)] = expr
        self._split_sym_products = new_split_prods

        new_cr_consumer = {}
        for cr_sid, consumer_sid in self._cache_read_consumer.items():
            new_cr = cr_sid + 1 if cr_sid >= inserted_stage_id else cr_sid
            new_con = consumer_sid + 1 if consumer_sid >= inserted_stage_id else consumer_sid
            new_cr_consumer[new_cr] = new_con
        self._cache_read_consumer = new_cr_consumer

        new_stencil = {}
        for cr_sid, info in self._cache_read_stencil_info.items():
            new_cr = cr_sid + 1 if cr_sid >= inserted_stage_id else cr_sid
            new_stencil[new_cr] = info
        self._cache_read_stencil_info = new_stencil

        new_shared = {}
        for sid, ext in self._shared_fused_extents.items():
            new_sid = sid + 1 if sid >= inserted_stage_id else sid
            new_shared[new_sid] = ext
        self._shared_fused_extents = new_shared

    # ─── 출력 ───
    def __str__(self):
        return self.to_str(delete_trivial_loop=False)

    def __repr__(self):
        return self.to_str(delete_trivial_loop=False)

    def to_str(self, delete_trivial_loop=True):
        lines = []
        placeholders = [s.op_name for s in self.stages if s.op_type == 'placeholder']
        lines.append("Placeholder: " + ", ".join(placeholders))
        for sid, stage in enumerate(self.stages):
            if stage.op_type == 'placeholder':
                continue
            if stage.compute_at == CA_ROOT:
                self._print_stage(lines, sid, 0, delete_trivial_loop)
        return "\n".join(lines)

    def _print_stage(self, lines, stage_id, base_indent, delete_trivial_loop):
        stage = self.stages[stage_id]
        if stage.auto_unroll_max_step is not None:
            lines.append(" " * base_indent + f"{stage.op_name} auto_unroll: {stage.auto_unroll_max_step}")
        if stage.storage_offset != 0:
            lines.append(" " * base_indent + f"{stage.op_name} storage_offset: {stage.storage_offset}")

        indent = 0
        for iid, it in enumerate(stage.iters):
            is_trivial = (it.extent is not None and it.extent.is_concrete and it.extent.val == 1)
            if not (delete_trivial_loop and is_trivial):
                ann = ANNOTATION_STR.get(it.annotation, "?")
                if it.extent is not None:
                    lines.append(" " * (base_indent + indent) + f"{ann} {it.name} (0,{it.extent})")
                else:
                    lines.append(" " * (base_indent + indent) + f"{ann} {it.name} (None)")
                indent += 2

            for asid, astage in enumerate(self.stages):
                if (astage.compute_at == CA_ITER and
                    astage.attach_stage_id == stage_id and
                    astage.attach_iter_id == iid):
                    self._print_stage(lines, asid, base_indent + indent, delete_trivial_loop)

        lines.append(" " * (base_indent + indent) + f"{stage.op_name} = ...")

    # ─── Symbolic extent 조회 함수 ───
    def _collect_extents_by_annotation(self, ann_codes):
        """주어진 annotation 코드 집합에 해당하는 iter의 (stage_id, iter_id, SymExpr) 목록 반환."""
        results = []
        for sid, stage in enumerate(self.stages):
            if stage.compute_at == CA_INLINED:
                continue
            for iid, it in enumerate(stage.iters):
                if it.annotation in ann_codes:
                    results.append((sid, iid, it.extent))
        return results

    def get_vectorize_extents(self):
        return self._collect_extents_by_annotation({2})

    def get_thread_extents(self):
        return self._collect_extents_by_annotation({6, 8, 10})

    def get_vthread_extents(self):
        return self._collect_extents_by_annotation({4})

    def get_shared_memory_extents(self):
        results = []
        for sid, ext in sorted(self._shared_fused_extents.items()):
            results.append((sid, self.stages[sid].op_name, ext))
        return results


# ─────────────────────────────────────────────────────────────
#  TransformApplier: transform step 적용 로직
# ─────────────────────────────────────────────────────────────
class TransformApplier:
    """
    SymbolicState에 transform steps를 순차 적용하는 로직.
    SymbolicState의 내부 데이터(stages, sym_map, 메타데이터)를 직접 조작.
    """

    def __init__(self, sym_state):
        """
        Args:
            sym_state: SymbolicState 인스턴스
        """
        self.s = sym_state  # SymbolicState 참조

    def apply_steps(self, state):
        """모든 transform steps를 순차 적용."""
        self.s._state = state
        steps = state.transform_steps
        for i, step in enumerate(steps):
            tk = step.type_key.split(".")[-1]
            if tk == "AnnotationStep":
                self._apply_annotation(step)
            elif tk == "FuseStep":
                self._apply_fuse(step, i)
            elif tk == "PragmaStep":
                self._apply_pragma(step, i)
            elif tk == "ReorderStep":
                self._apply_reorder(step)
            elif tk == "SplitStep":
                self._apply_split(step, i)
            elif tk == "FollowSplitStep":
                self._apply_follow_split(step, steps, i)
            elif tk == "FollowFusedSplitStep":
                self._apply_follow_fused_split(step, steps, i)
            elif tk == "StorageAlignStep":
                self._apply_storage_align(step)
            elif tk == "ComputeAtStep":
                self._apply_compute_at(step)
            elif tk == "ComputeInlineStep":
                self._apply_compute_inline(step)
            elif tk == "ComputeRootStep":
                self._apply_compute_root(step)
            elif tk == "CacheReadStep":
                self._apply_cache_read(step, state, i)
            elif tk == "CacheWriteStep":
                self._apply_cache_write(step, state, i)
            else:
                print(f"  [WARN] Unhandled step type: {tk}")

        self._infer_bound_final(state)

    # ─── Lazy extent 복원 ───
    def _restore_stage_extents_if_needed(self, stage_id, step_idx):
        s = self.s
        stage = s.stages[stage_id]
        has_none = any(it.extent is None for it in stage.iters)
        if not has_none:
            return

        ps = tvm._ffi.get_global_func("auto_scheduler.ReplayStepsPartial")(
            s.compute_dag, s._state, step_idx)
        bounded = s.compute_dag.infer_bound_from_state(ps)

        if stage_id >= len(bounded.stages):
            return

        cr_sym_candidates = None
        cr_stencil = None
        cr_ordered_splits = None
        if stage_id in s._cache_read_consumer:
            cr_sym_candidates = self._get_consumer_split_sym_products(stage_id)
            cr_stencil = s._cache_read_stencil_info.get(stage_id)
            if cr_stencil:
                consumer_sid = s._cache_read_consumer[stage_id]
                ordered = [(si, prod) for (sid, si), prod in s._split_sym_products.items()
                          if sid == consumer_sid]
                ordered.sort(key=lambda x: x[0])
                cr_ordered_splits = [(SymExpr(prod.val), eval_sym_extent(prod, s.sym_map))
                                    for si, prod in ordered]

        real_stage = bounded.stages[stage_id]
        for iid in range(len(stage.iters)):
            if stage.iters[iid].extent is None and iid < len(real_stage.iters):
                real_it = real_stage.iters[iid]
                if real_it.range is not None:
                    real_ext = int(real_it.range.extent)
                    saved = s._ca_saved_extents.get((stage_id, iid))
                    if saved is not None:
                        stage.iters[iid].extent = saved
                        continue
                    if cr_sym_candidates is not None:
                        if cr_stencil and cr_ordered_splits and iid in cr_stencil:
                            stride, sp_order, rd_order = cr_stencil[iid]
                            if stride == 0:
                                order = sp_order if sp_order is not None else rd_order
                                if order is not None and order < len(cr_ordered_splits):
                                    sym_expr, eval_val = cr_ordered_splits[order]
                                    stage.iters[iid].extent = sym_expr
                                    for ci_idx, (ev, se) in enumerate(cr_sym_candidates):
                                        if ev == eval_val and se.val == sym_expr.val:
                                            cr_sym_candidates.pop(ci_idx)
                                            break
                                    continue
                            elif stride > 0 and sp_order is not None and rd_order is not None:
                                sp_sym, sp_eval = cr_ordered_splits[sp_order]
                                rd_sym, rd_eval = cr_ordered_splits[rd_order]
                                predicted = (sp_eval - 1) * stride + rd_eval
                                if predicted == real_ext:
                                    stencil_expr = SymExpr(f"({sp_sym.val} - 1)*{stride} + {rd_sym.val}")
                                    stage.iters[iid].extent = stencil_expr
                                    continue
                        matched_sym = self._match_cr_extent(real_ext, cr_sym_candidates)
                        if matched_sym is not None:
                            stage.iters[iid].extent = matched_sym
                            cr_sym_candidates.remove((real_ext, matched_sym))
                            continue
                    # compute_at된 stage: target inner iter와 매칭 시도
                    ca_match = self._match_compute_at_inner_extent(stage_id, real_ext)
                    if ca_match is not None:
                        stage.iters[iid].extent = ca_match
                        continue
                    stage.iters[iid].extent = SymExpr(real_ext)

    def _get_consumer_split_sym_products(self, cache_read_stage_id):
        s = self.s
        consumer_sid = s._cache_read_consumer.get(cache_read_stage_id)
        if consumer_sid is None:
            return None
        candidates = []
        for (sid, step_idx), sym_prod in s._split_sym_products.items():
            if sid == consumer_sid:
                eval_val = eval_sym_extent(sym_prod, s.sym_map)
                if isinstance(eval_val, int):
                    candidates.append((eval_val, SymExpr(sym_prod.val)))
        return candidates

    @staticmethod
    def _match_cr_extent(real_ext, candidates):
        for eval_val, sym_expr in candidates:
            if eval_val == real_ext:
                return sym_expr
        return None

    def _match_compute_at_inner_extent(self, sid, real_ext):
        """compute_at된 stage의 iter에 대해, target stage의 inner iter 중
        eval(extent)==real_ext 인 non-concrete symbolic extent를 찾아 반환.
        매칭 실패 시 None."""
        s = self.s
        stage = s.stages[sid]
        if stage.compute_at != CA_ITER or stage.attach_stage_id is None:
            return None
        target_sid = stage.attach_stage_id
        target_iid = stage.attach_iter_id
        target_stage = s.stages[target_sid]
        # attach_iter_id 이후의 inner iters를 순회
        for tiid in range(target_iid + 1, len(target_stage.iters)):
            t_it = target_stage.iters[tiid]
            if t_it.extent is None or t_it.extent.is_concrete:
                continue
            t_eval = eval_sym_extent(t_it.extent, s.sym_map)
            if isinstance(t_eval, int) and t_eval == real_ext:
                return SymExpr(t_it.extent.val)
        return None

    def _infer_bound_final(self, state):
        s = self.s
        from tvm.auto_scheduler.loop_state import StateObject
        state_obj = state if isinstance(state, StateObject) else state.state_object
        bounded = s.compute_dag.infer_bound_from_state(state_obj)

        for sid in range(len(s.stages)):
            sym_stage = s.stages[sid]
            if sid >= len(bounded.stages):
                continue
            real_stage = bounded.stages[sid]

            cr_sym_candidates = None
            cr_stencil = None
            cr_ordered_splits = None
            if sid in s._cache_read_consumer:
                cr_sym_candidates = self._get_consumer_split_sym_products(sid)
                cr_stencil = s._cache_read_stencil_info.get(sid)
                if cr_stencil:
                    consumer_sid = s._cache_read_consumer[sid]
                    ordered = [(si, prod) for (s2, si), prod in s._split_sym_products.items()
                              if s2 == consumer_sid]
                    ordered.sort(key=lambda x: x[0])
                    cr_ordered_splits = [(SymExpr(prod.val), eval_sym_extent(prod, s.sym_map))
                                        for si, prod in ordered]

            for iid in range(len(sym_stage.iters)):
                sym_it = sym_stage.iters[iid]
                if sym_it.extent is None and iid < len(real_stage.iters):
                    real_it = real_stage.iters[iid]
                    if real_it.range is None:
                        continue
                    real_ext = int(real_it.range.extent)
                    saved = s._ca_saved_extents.get((sid, iid))
                    if saved is not None:
                        sym_it.extent = saved
                        continue
                    if cr_sym_candidates is not None:
                        if cr_stencil and cr_ordered_splits and iid in cr_stencil:
                            stride, sp_order, rd_order = cr_stencil[iid]
                            if stride == 0:
                                order = sp_order if sp_order is not None else rd_order
                                if order is not None and order < len(cr_ordered_splits):
                                    sym_expr, eval_val = cr_ordered_splits[order]
                                    sym_it.extent = sym_expr
                                    for ci_idx, (ev, se) in enumerate(cr_sym_candidates):
                                        if ev == eval_val and se.val == sym_expr.val:
                                            cr_sym_candidates.pop(ci_idx)
                                            break
                                    continue
                            elif stride > 0 and sp_order is not None and rd_order is not None:
                                sp_sym, sp_eval = cr_ordered_splits[sp_order]
                                rd_sym, rd_eval = cr_ordered_splits[rd_order]
                                predicted = (sp_eval - 1) * stride + rd_eval
                                if predicted == real_ext:
                                    stencil_expr = SymExpr(f"({sp_sym.val} - 1)*{stride} + {rd_sym.val}")
                                    sym_it.extent = stencil_expr
                                    continue
                        matched_sym = self._match_cr_extent(real_ext, cr_sym_candidates)
                        if matched_sym is not None:
                            sym_it.extent = matched_sym
                            cr_sym_candidates.remove((real_ext, matched_sym))
                            continue
                    # compute_at된 stage: target inner iter와 매칭 시도
                    ca_match = self._match_compute_at_inner_extent(sid, real_ext)
                    if ca_match is not None:
                        sym_it.extent = ca_match
                        continue
                    sym_it.extent = SymExpr(real_ext)

    # ─── AnnotationStep ───
    def _apply_annotation(self, step):
        self.s.stages[step.stage_id].iters[step.iter_id].annotation = int(step.annotation)

    # ─── FuseStep ───
    def _apply_fuse(self, step, step_idx):
        s = self.s
        sid = step.stage_id
        fused_ids = [int(x) for x in step.fused_ids]
        stage = s.stages[sid]

        if not fused_ids:
            new_it = SymIter("", SymExpr(1), annotation=0, iter_kind=3)
            stage.iters.insert(0, new_it)
            return

        self._restore_stage_extents_if_needed(sid, step_idx)

        new_name = "@".join(stage.iters[fid].name for fid in fused_ids) + "@"

        new_extent = SymExpr(1)
        all_defined = True
        new_iter_kind = stage.iters[fused_ids[0]].iter_kind
        for fid in fused_ids:
            it = stage.iters[fid]
            if it.extent is not None:
                new_extent = SymExpr.mul(new_extent, it.extent)
            else:
                all_defined = False
            if it.iter_kind != new_iter_kind:
                new_iter_kind = 2

        new_it = SymIter(new_name,
                         new_extent if all_defined else None,
                         annotation=0, iter_kind=new_iter_kind)

        if all_defined and ".shared" in stage.op_name:
            s._shared_fused_extents[sid] = SymExpr(new_extent.val)

        begin = fused_ids[0]
        end = fused_ids[-1]
        new_iters = stage.iters[:begin] + [new_it] + stage.iters[end + 1:]
        stage.iters = new_iters

        removed = len(fused_ids) - 1
        for other_stage in s.stages:
            if (other_stage.compute_at == CA_ITER and
                other_stage.attach_stage_id == sid):
                old_aid = other_stage.attach_iter_id
                if old_aid > end:
                    other_stage.attach_iter_id = old_aid - removed
                elif old_aid >= begin:
                    other_stage.attach_iter_id = begin

    # ─── PragmaStep ───
    def _apply_pragma(self, step, step_idx):
        s = self.s
        sid = step.stage_id
        pragma_type = str(step.pragma_type)
        if pragma_type.startswith("auto_unroll_max_step"):
            parts = pragma_type.split("$")
            if len(parts) == 2:
                val = int(parts[1])
                sym_name = f"ur_{step_idx}"
                s.sym_map[sym_name] = val
                s.stages[sid].auto_unroll_max_step = SymExpr(sym_name)
        elif pragma_type == "debug_skip_region":
            s.stages[sid].compute_at = CA_ROOT
            s.stages[sid].attach_stage_id = None
            s.stages[sid].attach_iter_id = None

    # ─── ReorderStep ───
    def _apply_reorder(self, step):
        s = self.s
        sid = step.stage_id
        after_ids = [int(x) for x in step.after_ids]
        stage = s.stages[sid]
        old_iters = stage.iters
        stage.iters = [old_iters[i] for i in after_ids]
        old_to_new = {old_id: new_id for new_id, old_id in enumerate(after_ids)}
        for other_stage in s.stages:
            if (other_stage.compute_at == CA_ITER and
                other_stage.attach_stage_id == sid and
                other_stage.attach_iter_id in old_to_new):
                other_stage.attach_iter_id = old_to_new[other_stage.attach_iter_id]

    # ─── SplitStep ───
    def _apply_split(self, step, step_idx):
        s = self.s
        sid = step.stage_id
        iid = step.iter_id
        lengths = list(step.lengths)
        inner_to_outer = bool(step.inner_to_outer)

        stage = s.stages[sid]
        self._restore_stage_extents_if_needed(sid, step_idx)

        orig_iter = stage.iters[iid]

        if orig_iter.extent is not None:
            tosplit_extent = orig_iter.extent
        elif step.extent is not None:
            tosplit_extent = SymExpr(int(step.extent))
        else:
            tosplit_extent = None

        sym_lengths = []
        for li, length in enumerate(lengths):
            val = int(length) if length is not None else None
            sym_name = f"sp_{step_idx}_{li}"
            s.sym_map[sym_name] = val
            sym_lengths.append(SymExpr(sym_name))

        sym_prod = SymExpr.product(sym_lengths)
        s._split_sym_products[(sid, step_idx)] = sym_prod

        outs = []
        if inner_to_outer:
            for i in range(len(lengths)):
                li = len(lengths) - i - 1
                name = f"{orig_iter.name}.{len(lengths) - i}"
                sym_ext = sym_lengths[li]
                if tosplit_extent is not None:
                    sym_ext = SymExpr.min(sym_ext, tosplit_extent)
                outs.append(SymIter(name, sym_ext, annotation=0, iter_kind=orig_iter.iter_kind))
                if tosplit_extent is not None:
                    tosplit_extent = SymExpr.ceildiv(tosplit_extent, sym_ext)
                else:
                    tosplit_extent = None
            outs.append(SymIter(f"{orig_iter.name}.0", tosplit_extent,
                                annotation=0, iter_kind=orig_iter.iter_kind))
            outs = list(reversed(outs))
        else:
            for i in range(len(lengths)):
                name = f"{orig_iter.name}.{i}"
                sym_ext = sym_lengths[i]
                if tosplit_extent is not None:
                    sym_ext = SymExpr.min(sym_ext, tosplit_extent)
                outs.append(SymIter(name, sym_ext, annotation=0, iter_kind=orig_iter.iter_kind))
                if tosplit_extent is not None:
                    tosplit_extent = SymExpr.ceildiv(tosplit_extent, sym_ext)
                else:
                    tosplit_extent = None
            outs.append(SymIter(f"{orig_iter.name}.{len(lengths)}", tosplit_extent,
                                annotation=0, iter_kind=orig_iter.iter_kind))

        new_iters = stage.iters[:iid] + outs + stage.iters[iid + 1:]
        stage.iters = new_iters

        shift = len(lengths)
        for other_stage in s.stages:
            if (other_stage.compute_at == CA_ITER and
                other_stage.attach_stage_id == sid and
                other_stage.attach_iter_id >= iid):
                other_stage.attach_iter_id += shift

    # ─── FollowSplitStep ───
    def _apply_follow_split(self, step, all_steps, step_idx):
        s = self.s
        sid = step.stage_id
        iid = step.iter_id
        src_step_id = step.src_step_id
        n_split = int(step.n_split)

        src_step = all_steps[src_step_id]
        src_lengths = list(src_step.lengths)

        extracted = []
        for j in range(min(n_split - 1, len(src_lengths))):
            extracted.append(src_lengths[j])
        last = 1
        j_start = n_split - 1
        all_defined = True
        for j in range(j_start, len(src_lengths)):
            if src_lengths[j] is not None:
                last *= int(src_lengths[j])
            else:
                all_defined = False
                break
        extracted.append(last if all_defined else None)

        stage = s.stages[sid]
        self._restore_stage_extents_if_needed(sid, step_idx)

        orig_iter = stage.iters[iid]
        orig_extent = None
        if orig_iter.extent is not None and orig_iter.extent.is_concrete:
            orig_extent = orig_iter.extent.val

        tosplit_extent = SymExpr(orig_extent) if orig_extent is not None else \
                         (orig_iter.extent if orig_iter.extent is not None else None)
        sym_lengths = []
        for li in range(len(extracted)):
            if li < n_split - 1:
                src_sym_name = f"sp_{src_step_id}_{li}"
            else:
                if j_start < len(src_lengths) and j_start == len(src_lengths) - 1:
                    src_sym_name = f"sp_{src_step_id}_{j_start}"
                else:
                    parts = [f"sp_{src_step_id}_{j}" for j in range(j_start, len(src_lengths))]
                    src_sym_name = "*".join(parts) if parts else str(extracted[li])
            sym_lengths.append(SymExpr(src_sym_name))

        outs = []
        for i in range(len(extracted)):
            li = len(extracted) - i - 1
            name = f"{orig_iter.name}.{len(extracted) - i}"
            sym_ext = sym_lengths[li]
            if tosplit_extent is not None:
                sym_ext = SymExpr.min(sym_ext, tosplit_extent)
            outs.append(SymIter(name, sym_ext, annotation=0, iter_kind=orig_iter.iter_kind))
            if tosplit_extent is not None:
                tosplit_extent = SymExpr.ceildiv(tosplit_extent, sym_ext)
            else:
                tosplit_extent = None
        outs.append(SymIter(f"{orig_iter.name}.0", tosplit_extent,
                            annotation=0, iter_kind=orig_iter.iter_kind))
        outs = list(reversed(outs))

        new_iters = stage.iters[:iid] + outs + stage.iters[iid + 1:]
        stage.iters = new_iters

        shift = len(extracted)
        for other_stage in s.stages:
            if (other_stage.compute_at == CA_ITER and
                other_stage.attach_stage_id == sid and
                other_stage.attach_iter_id >= iid):
                other_stage.attach_iter_id += shift

    # ─── FollowFusedSplitStep ───
    def _apply_follow_fused_split(self, step, all_steps, step_idx):
        s = self.s
        sid = step.stage_id
        iid = step.iter_id
        src_step_ids = [int(x) for x in step.src_step_ids]
        level = int(step.level)
        factor_or_nparts = bool(step.factor_or_nparts)

        stage = s.stages[sid]
        self._restore_stage_extents_if_needed(sid, step_idx)

        orig_iter = stage.iters[iid]
        orig_extent = None
        if orig_iter.extent is not None and orig_iter.extent.is_concrete:
            orig_extent = orig_iter.extent.val

        tosplit_extent = SymExpr(orig_extent) if orig_extent is not None else \
                         (orig_iter.extent if orig_iter.extent is not None else None)

        src_sym_parts = []
        for sid_ref in src_step_ids:
            src_sym_parts.append(f"sp_{sid_ref}_{level}")
        fused_sym_expr = SymExpr("*".join(src_sym_parts) if len(src_sym_parts) > 1 else src_sym_parts[0])

        if factor_or_nparts:
            inner_ext = fused_sym_expr
            outer_ext = SymExpr.ceildiv(tosplit_extent, fused_sym_expr) if tosplit_extent else None
            outs = [
                SymIter(f"{orig_iter.name}.0", outer_ext, annotation=0, iter_kind=orig_iter.iter_kind),
                SymIter(f"{orig_iter.name}.1", inner_ext, annotation=0, iter_kind=orig_iter.iter_kind),
            ]
        else:
            outer_ext = fused_sym_expr
            inner_ext = SymExpr.ceildiv(tosplit_extent, fused_sym_expr) if tosplit_extent else None
            outs = [
                SymIter(f"{orig_iter.name}.0", outer_ext, annotation=0, iter_kind=orig_iter.iter_kind),
                SymIter(f"{orig_iter.name}.1", inner_ext, annotation=0, iter_kind=orig_iter.iter_kind),
            ]

        new_iters = stage.iters[:iid] + outs + stage.iters[iid + 1:]
        stage.iters = new_iters

        for other_stage in s.stages:
            if (other_stage.compute_at == CA_ITER and
                other_stage.attach_stage_id == sid and
                other_stage.attach_iter_id >= iid):
                other_stage.attach_iter_id += 1

    # ─── StorageAlignStep ───
    def _apply_storage_align(self, step):
        self.s.stages[step.stage_id].storage_offset = step.offset

    # ─── ComputeAtStep ───
    def _apply_compute_at(self, step):
        s = self.s
        sid = step.stage_id
        target_sid = step.target_stage_id
        target_iid = step.target_iter_id
        stage = s.stages[sid]

        for iid, it in enumerate(stage.iters):
            if it.extent is not None and not it.extent.is_concrete:
                s._ca_saved_extents[(sid, iid)] = SymExpr(it.extent.val)
            it.extent = None

        stage.compute_at = CA_ITER
        stage.attach_stage_id = target_sid
        stage.attach_iter_id = target_iid

    # ─── ComputeInlineStep ───
    def _apply_compute_inline(self, step):
        stage = self.s.stages[step.stage_id]
        stage.compute_at = CA_INLINED
        stage.attach_stage_id = None
        stage.attach_iter_id = None

    # ─── ComputeRootStep ───
    def _apply_compute_root(self, step):
        s = self.s
        sid = step.stage_id
        stage = s.stages[sid]
        for iid, it in enumerate(stage.iters):
            if it.extent is not None and not it.extent.is_concrete:
                s._ca_saved_extents[(sid, iid)] = SymExpr(it.extent.val)
            it.extent = None
        stage.compute_at = CA_ROOT
        stage.attach_stage_id = None
        stage.attach_iter_id = None

    # ─── CacheReadStep ───
    def _apply_cache_read(self, step, state, step_idx):
        s = self.s
        sid = step.stage_id
        added_stage_id = sid + 1

        ps_after = tvm._ffi.get_global_func("auto_scheduler.ReplayStepsPartial")(
            s.compute_dag, state, step_idx + 1)

        new_stage_real = ps_after.stages[added_stage_id]
        new_iters = []
        for it in new_stage_real.iters:
            ext = int(it.range.extent) if it.range is not None else None
            new_iters.append(SymIter(it.name, SymExpr(ext) if ext is not None else None,
                                      annotation=0, iter_kind=0))

        new_sym_stage = SymStage(new_stage_real.op.name, 'compute', new_iters,
                                 dtype=str(new_stage_real.op.output(0).dtype)
                                       if hasattr(new_stage_real.op, 'output') else "float32")
        s.stages.insert(added_stage_id, new_sym_stage)

        s._shift_ca_saved_extents(added_stage_id)

        reader_ids = [int(x) for x in step.reader_stage_ids]
        if reader_ids:
            consumer_sid = reader_ids[0]
            if consumer_sid >= added_stage_id:
                consumer_sid += 1
            s._cache_read_consumer[added_stage_id] = consumer_sid
            self._analyze_cache_read_stencil(added_stage_id, sid, consumer_sid)

        for other_stage in s.stages:
            if other_stage.compute_at == CA_ITER:
                if other_stage.attach_stage_id is not None and other_stage.attach_stage_id >= added_stage_id:
                    other_stage.attach_stage_id += 1

        for i in range(added_stage_id + 1, len(s.stages)):
            real_stage = ps_after.stages[i]
            s.stages[i].op_name = real_stage.op.name

    def _analyze_cache_read_stencil(self, cr_stage_id, orig_tensor_sid, consumer_sid):
        s = self.s
        from tvm import tir

        cr_name = s.stages[cr_stage_id].op_name
        orig_name = cr_name.rsplit(".", 1)[0] if "." in cr_name else cr_name

        consumer_name = s.stages[consumer_sid].op_name
        consumer_orig_name = consumer_name.rsplit(".", 1)[0] if "." in consumer_name else consumer_name

        consumer_op = None
        for op in s.compute_dag.ops:
            if op.name == consumer_orig_name or op.name == consumer_name:
                if hasattr(op, 'body'):
                    consumer_op = op
                    break

        if consumer_op is None or not hasattr(consumer_op, 'body') or len(consumer_op.body) == 0:
            return

        producer_loads = self._find_producer_loads(consumer_op.body[0], orig_name)
        if not producer_loads:
            return

        pl = producer_loads[0]

        spatial_vars = {str(ax.var.name): i for i, ax in enumerate(consumer_op.axis)}
        reduce_vars = {str(ax.var.name): len(consumer_op.axis) + i
                      for i, ax in enumerate(consumer_op.reduce_axis)}

        stencil_info = {}
        for ax_idx, idx_expr in enumerate(pl.indices):
            result = self._analyze_index_expr(idx_expr, consumer_op.axis, consumer_op.reduce_axis)
            if result is None:
                continue
            stride, sp_name, rd_name = result
            sp_order = spatial_vars.get(sp_name) if sp_name else None
            rd_order = reduce_vars.get(rd_name) if rd_name else None
            stencil_info[ax_idx] = (stride, sp_order, rd_order)

        if stencil_info:
            s._cache_read_stencil_info[cr_stage_id] = stencil_info

    @staticmethod
    def _find_producer_loads(expr, tensor_name):
        from tvm import tir
        results = []
        def visit(e):
            if isinstance(e, tir.ProducerLoad):
                if str(e.producer.name) == tensor_name:
                    results.append(e)
                return
            if isinstance(e, tir.Reduce):
                for src in e.source:
                    visit(src)
                return
            for attr in ['a', 'b']:
                if hasattr(e, attr):
                    visit(getattr(e, attr))
            if hasattr(e, 'args') and not isinstance(e, (tir.Var, tir.IntImm)):
                for arg in e.args:
                    visit(arg)
        visit(expr)
        return results

    @staticmethod
    def _analyze_index_expr(expr, spatial_axes, reduce_axes):
        from tvm import tir
        spatial_names = {str(v.var.name) for v in spatial_axes}
        reduce_names = {str(v.var.name) for v in reduce_axes}

        if isinstance(expr, tir.Var):
            name = str(expr.name)
            if name in spatial_names:
                return (0, name, None)
            elif name in reduce_names:
                return (0, None, name)
            return None

        if isinstance(expr, tir.Add):
            a, b = expr.a, expr.b
            def extract_mul_var(e):
                if isinstance(e, tir.Mul):
                    if isinstance(e.a, tir.Var) and isinstance(e.b, tir.IntImm):
                        return (str(e.a.name), int(e.b))
                    if isinstance(e.b, tir.Var) and isinstance(e.a, tir.IntImm):
                        return (str(e.b.name), int(e.a))
                elif isinstance(e, tir.Var):
                    return (str(e.name), 1)
                return None

            mul_info = extract_mul_var(a)
            if mul_info and isinstance(b, tir.Var):
                sp_name, stride = mul_info
                rd_name = str(b.name)
                if sp_name in spatial_names and rd_name in reduce_names:
                    return (stride, sp_name, rd_name)

            mul_info = extract_mul_var(b)
            if mul_info and isinstance(a, tir.Var):
                sp_name, stride = mul_info
                rd_name = str(a.name)
                if sp_name in spatial_names and rd_name in reduce_names:
                    return (stride, sp_name, rd_name)

            if isinstance(a, tir.Var) and isinstance(b, tir.Var):
                a_name, b_name = str(a.name), str(b.name)
                if a_name in spatial_names and b_name in reduce_names:
                    return (1, a_name, b_name)
                if b_name in spatial_names and a_name in reduce_names:
                    return (1, b_name, a_name)

        return None

    # ─── CacheWriteStep ───
    def _apply_cache_write(self, step, state, step_idx):
        s = self.s
        sid = step.stage_id
        ps_after = tvm._ffi.get_global_func("auto_scheduler.ReplayStepsPartial")(
            s.compute_dag, state, step_idx + 1)

        new_stage_real = ps_after.stages[sid]
        new_iters = []
        for it in new_stage_real.iters:
            ext = int(it.range.extent) if it.range is not None else None
            new_iters.append(SymIter(it.name, SymExpr(ext) if ext is not None else None,
                                      annotation=0, iter_kind=0))

        new_sym_stage = SymStage(new_stage_real.op.name, 'compute', new_iters,
                                 dtype=str(new_stage_real.op.output(0).dtype)
                                       if hasattr(new_stage_real.op, 'output') else "float32")
        s.stages.insert(sid, new_sym_stage)

        s._shift_ca_saved_extents(sid)

        for other_stage in s.stages:
            if other_stage.compute_at == CA_ITER:
                if other_stage.attach_stage_id is not None and other_stage.attach_stage_id >= sid:
                    other_stage.attach_stage_id += 1

        for i in range(len(s.stages)):
            if i < len(ps_after.stages):
                real_stage = ps_after.stages[i]
                s.stages[i].op_name = real_stage.op.name


# ─────────────────────────────────────────────────────────────
#  SymParamManager: 파라미터 조회 / 검증 / 랜덤 생성
# ─────────────────────────────────────────────────────────────
class SymParamManager:
    """
    SymbolicState의 sym_map 파라미터를 조회·검증·수정하는 매니저.
    """

    UNROLL_CANDIDATES = [0, 16, 64, 512, 1024]

    def __init__(self, sym_state):
        """
        Args:
            sym_state: SymbolicState 인스턴스
        """
        self.s = sym_state

    def _build_sp_groups(self):
        """SP 그룹 구성: step_idx → [sym_name, ...] (length_idx 순 정렬)."""
        sp_groups = {}
        for name in self.s.sym_map:
            if name.startswith("sp_"):
                parts = name.split("_")
                step_idx = int(parts[1])
                sp_groups.setdefault(step_idx, []).append(name)
        for step_idx in sp_groups:
            sp_groups[step_idx].sort(key=lambda n: int(n.split("_")[2]))
        return sp_groups

    def _build_sp_extents(self, sp_groups):
        """SP step_idx → extent 매핑 (SplitStep의 원본 extent)."""
        sp_extents = {}
        if self.s._state is not None:
            steps = self.s._state.transform_steps
            for step_idx in sp_groups:
                if step_idx < len(steps):
                    step = steps[step_idx]
                    tk = step.type_key.split(".")[-1]
                    if tk == "SplitStep" and step.extent is not None:
                        sp_extents[step_idx] = int(step.extent)
        return sp_extents

    @staticmethod
    def _divisors(n):
        """양의 정수 n의 약수 목록을 오름차순으로 반환."""
        if n <= 0:
            return [1]
        divs = []
        for i in range(1, int(n**0.5) + 1):
            if n % i == 0:
                divs.append(i)
                if i != n // i:
                    divs.append(n // i)
        return sorted(divs)


# ─────────────────────────────────────────────────────────────
#  편의 함수: SymbolicState 생성 + TransformApplier 적용 (원스텝)
# ─────────────────────────────────────────────────────────────
def build_symbolic_state(compute_dag, state):
    """compute_dag + state로부터 SymbolicState를 생성하고 steps를 적용.
    Returns: SymbolicState (apply 완료)
    """
    sym = SymbolicState(compute_dag)
    applier = TransformApplier(sym)
    applier.apply_steps(state)
    return sym


# ─────────────────────────────────────────────────────────────
#  검증 유틸리티
# ─────────────────────────────────────────────────────────────
def verify_symbolic_state(task, state, sym_state, verbose=False):
    """
    다른 state의 구체적 파라미터를 sym_map에 반영한 뒤,
    InferBound 결과와 SymbolicState 구조를 비교한다.

    같은 sketch의 state들은 구조(stage/iter 이름, annotation)는 동일하고
    split factor와 unroll 값만 다르다.
    → 검증 대상 state의 split/unroll 값으로 sym_map을 교체한 뒤 eval.

    Returns: (ok: bool, summary: str)
    """
    # 검증 대상 state의 SplitStep/PragmaStep 값을 추출하여 sym_map에 반영
    saved_sym_map = dict(sym_state.sym_map)  # 원본 백업
    for step_idx, step in enumerate(state.transform_steps):
        tk = step.type_key.split(".")[-1]
        if tk == "SplitStep":
            for li, length in enumerate(step.lengths):
                sym_name = f"sp_{step_idx}_{li}"
                if sym_name in sym_state.sym_map:
                    sym_state.sym_map[sym_name] = int(length) if length is not None else None
        elif tk == "PragmaStep":
            pragma_type = str(step.pragma_type)
            if "auto_unroll_max_step$" in pragma_type:
                val = int(pragma_type.split("$")[1])
                sym_name = f"ur_{step_idx}"
                if sym_name in sym_state.sym_map:
                    sym_state.sym_map[sym_name] = val

    bounded = task.compute_dag.infer_bound_from_state(state)

    stage_mismatch = []
    name_mm = 0
    ann_mm = 0
    ext_mm = 0
    ext_total = 0
    details = []

    n_stages = min(len(bounded.stages), len(sym_state.stages))
    if len(bounded.stages) != len(sym_state.stages):
        details.append(f"Stage count: real={len(bounded.stages)} sym={len(sym_state.stages)}")

    for sid in range(n_stages):
        rs = bounded.stages[sid]
        ss = sym_state.stages[sid]
        if len(rs.iters) != len(ss.iters):
            stage_mismatch.append((sid, len(rs.iters), len(ss.iters)))
            continue
        for iid in range(len(rs.iters)):
            ri, si = rs.iters[iid], ss.iters[iid]
            if str(ri.name) != si.name:
                name_mm += 1
                if verbose and name_mm <= 5:
                    details.append(f"  NAME s{sid}.i{iid}: real='{ri.name}' sym='{si.name}'")
            if int(ri.annotation) != si.annotation:
                ann_mm += 1
                if verbose and ann_mm <= 5:
                    details.append(f"  ANN  s{sid}.i{iid}: real={int(ri.annotation)} sym={si.annotation}")
            re_ext = int(ri.range.extent) if ri.range is not None else None
            se_ext = eval_sym_extent(si.extent, sym_state.sym_map)
            ext_total += 1
            if re_ext is not None and se_ext is not None:
                if se_ext < re_ext:
                    ext_mm += 1
                    if verbose and ext_mm <= 5:
                        details.append(f"  EXT  s{sid}.i{iid}('{si.name}'): real={re_ext} sym={si.extent}→eval={se_ext}")
            elif re_ext != se_ext:
                ext_mm += 1
                if verbose and ext_mm <= 5:
                    details.append(f"  EXT  s{sid}.i{iid}('{si.name}'): real={re_ext} sym={si.extent}→eval={se_ext}")

    # sym_map 원복
    sym_state.sym_map = saved_sym_map

    ok = (len(stage_mismatch) == 0 and name_mm == 0 and ann_mm == 0 and ext_mm == 0
          and len(bounded.stages) == len(sym_state.stages))
    parts = []
    if stage_mismatch:
        parts.append(f"iter_count_mm={len(stage_mismatch)}")
    if name_mm:
        parts.append(f"name_mm={name_mm}")
    if ann_mm:
        parts.append(f"ann_mm={ann_mm}")
    if ext_mm:
        parts.append(f"ext_mm={ext_mm}/{ext_total}")
    summary = "PASS" if ok else "FAIL(" + ", ".join(parts) + ")"
    if verbose and details:
        summary += "\n" + "\n".join(details)
    return ok, summary

In [35]:
from collections import defaultdict
from tvm.auto_scheduler.measure_record import load_records

def clean_name(x):
    x = str(x)
    x = x.replace(" ", "")
    x = x.replace('"', '')
    x = x.replace("'", '')
    return x

def get_task_json_name(records_dir, task):
    task_key = (task.workload_key, str(task.target.kind))
    return f"{records_dir}/{clean_name(task_key)}.json"


# ─────────────────────────────────────────────
# sketch fingerprint: 구조적 속성으로 sketch 식별
# ─────────────────────────────────────────────
def _step_structural_fingerprint(step):
    """step의 구조적 속성만 추출하여 hashable tuple 반환.
    SplitStep의 lengths 값, PragmaStep의 unroll 값은 제외 (이것들이 파라미터)."""
    tk = step.type_key.split(".")[-1]
    if tk == "AnnotationStep":
        return (tk, int(step.stage_id), int(step.iter_id), int(step.annotation))
    elif tk == "FuseStep":
        return (tk, int(step.stage_id), tuple(int(x) for x in step.fused_ids))
    elif tk == "PragmaStep":
        ptype = str(step.pragma_type).split("$")[0]
        return (tk, int(step.stage_id), int(step.iter_id), ptype)
    elif tk == "ReorderStep":
        return (tk, int(step.stage_id), tuple(int(x) for x in step.after_ids))
    elif tk == "SplitStep":
        return (tk, int(step.stage_id), int(step.iter_id),
                len(step.lengths), bool(step.inner_to_outer))
    elif tk == "FollowSplitStep":
        return (tk, int(step.stage_id), int(step.iter_id),
                int(step.src_step_id), int(step.n_split))
    elif tk == "FollowFusedSplitStep":
        return (tk, int(step.stage_id), int(step.iter_id),
                tuple(int(x) for x in step.src_step_ids),
                int(step.level), bool(step.factor_or_nparts))
    elif tk == "StorageAlignStep":
        return (tk, int(step.stage_id), int(step.iter_id),
                int(step.factor), int(step.offset))
    elif tk == "ComputeAtStep":
        return (tk, int(step.stage_id), int(step.target_stage_id), int(step.target_iter_id))
    elif tk == "ComputeInlineStep":
        return (tk, int(step.stage_id))
    elif tk == "ComputeRootStep":
        return (tk, int(step.stage_id))
    elif tk == "CacheReadStep":
        return (tk, int(step.stage_id), str(step.scope_name),
                tuple(int(x) for x in step.reader_stage_ids))
    elif tk == "CacheWriteStep":
        return (tk, int(step.stage_id), str(step.scope_name))
    else:
        return (tk, int(step.stage_id))

def state_sketch_fingerprint(state):
    """state의 전체 step 시퀀스에서 구조적 fingerprint를 추출.
    split factor 값과 pragma 값을 제외한 모든 구조 속성이 같아야 같은 sketch."""
    return tuple(_step_structural_fingerprint(s) for s in state.transform_steps)


# ─────────────────────────────────────────────
# 레코드 로드
# ─────────────────────────────────────────────
def load_records_from_dir(tasks, records_dir):
    """각 task에 대응하는 json 파일에서 레코드를 로드.
    Returns: dict {workload_key: [(MeasureInput, MeasureResult), ...]}
    """
    records = {}
    for task in tasks:
        json_path = get_task_json_name(records_dir, task)
        if os.path.exists(json_path):
            recs = load_records(json_path)
            records[task.workload_key] = [(mi, mr) for mi, mr in recs]
    return records


# ─────────────────────────────────────────────
# wkey → sketch 2단계 그룹핑
# ─────────────────────────────────────────────
def group_records_by_wkey_and_sketch(records):
    """레코드를 workload_key → sketch(step type 시퀀스) 기준으로 2단계 그룹핑.

    같은 task(=wkey)라도 transform step 종류의 시퀀스가 다르면 다른 sketch로 분류.
    step의 구체적인 속성(stage_id, split factor, pragma 값 등)은 무시한다.

    Returns:
        grouped: dict {wkey: {sketch_fp: [(MeasureInput, MeasureResult), ...]}}
        여기서 sketch_fp는 step type_key 이름들의 tuple.
    """
    grouped = {}   # wkey -> {sketch_fp -> [(inp, res), ...]}
    for wkey, recs in records.items():
        sketch_groups = defaultdict(list)
        for inp, res in recs:
            fp = state_sketch_fingerprint(inp.state)
            sketch_groups[fp].append((inp, res))
        grouped[wkey] = dict(sketch_groups)
    return grouped

def group_by_sketches_from_json(tasks, records_dir, verbose=False):
    """json 파일에서 레코드를 로드하여 sketch 기준으로 그룹핑한 결과 반환."""
    records = load_records_from_dir(tasks, records_dir)
    grouped = group_records_by_wkey_and_sketch(records)

    if verbose:
        wkey_to_task = {t.workload_key: t for t in tasks}

        for wkey, sketch_dict in grouped.items():
            task = wkey_to_task.get(wkey)
            desc = task.desc if task else wkey
            total_recs = sum(len(v) for v in sketch_dict.values())
            print(f"\n[{desc}] total records: {total_recs}, sketches: {len(sketch_dict)}")
    
    return grouped

json_dir = f"/root/work/tvm-ansor/gallery/dataset/to_measure_programs/((llama_tiny,[(1,64)]),cuda)"
grouped = group_by_sketches_from_json(tasks, json_dir, verbose=False)

In [39]:
# ═══════════════════════════════════════════════════════════════
# 종합 검증: 모든 task × 여러 state에서 SymbolicState 검증
# ═══════════════════════════════════════════════════════════════
import time


total_pass = 0
total_fail = 0
fail_details = []

t0 = time.time()



for task_idx, wkey in enumerate(grouped.keys()):
    
    task = tasks[task_idx]
    for sketch_idx, sketch_fp in enumerate(grouped[wkey].keys()):
        print(f"Testing Task {task_idx+1}/{len(tasks)}, Sketch {sketch_idx+1}/{len(grouped[wkey])}...")
        try:
            states = [inp.state for inp, res in grouped[wkey][sketch_fp]]
                    
        except Exception as e:
            print(f"T{task_idx+1} S{sketch_idx+1} [{task.desc}]  ⚠️  sample failed: {e}")
            continue

        task_pass = 0
        task_fail = 0
        task_fail_msgs = []

        sym_state = build_symbolic_state(task.compute_dag, states[0])

        for si in range(len(states)):
            state = states[si]
            try:
                ok, summary = verify_symbolic_state(task, state, sym_state, verbose=True)
            except Exception as e:
                ok = False
                summary = f"EXCEPTION: {e}"

            if ok:
                task_pass += 1
            else:
                task_fail += 1
                task_fail_msgs.append(f"    state[{si}]: {summary}")

        total_pass += task_pass
        total_fail += task_fail

        status = "✅" if task_fail == 0 else "❌"
        print(f"T{task_idx:2d} S{sketch_idx:2d} {status} {task_pass}/{len(states)} pass  [{task.desc}]")
        if task_fail_msgs:
            for msg in task_fail_msgs:
                print(msg)
            fail_details.extend(task_fail_msgs)

elapsed = time.time() - t0

print()
print("=" * 70)
print(f"Total: {total_pass + total_fail} tests, "
      f"{total_pass} passed, {total_fail} failed  "
      f"({elapsed:.1f}s)")
if total_fail == 0:
    print("✅✅✅ ALL TESTS PASSED! ✅✅✅")
else:
    print(f"❌ {total_fail} failures")
print("=" * 70)

Testing Task 1/6, Sketch 1/1...
T 0 S 0 ✅ 2000/2000 pass  [vm_mod_fused_nn_batch_matmul_3]
Testing Task 2/6, Sketch 1/1...
T 0 S 0 ✅ 2000/2000 pass  [vm_mod_fused_nn_batch_matmul_3]
Testing Task 2/6, Sketch 1/1...
T 1 S 0 ✅ 2000/2000 pass  [vm_mod_fused_nn_batch_matmul_1]

Total: 4000 tests, 4000 passed, 0 failed  (34.5s)
✅✅✅ ALL TESTS PASSED! ✅✅✅
T 1 S 0 ✅ 2000/2000 pass  [vm_mod_fused_nn_batch_matmul_1]

Total: 4000 tests, 4000 passed, 0 failed  (34.5s)
✅✅✅ ALL TESTS PASSED! ✅✅✅


In [22]:

# ─────────────────────────────────────────────────────────────
#  ExprNode: 제약식 표현식 트리 (interval arithmetic 지원)
# ─────────────────────────────────────────────────────────────
class ExprNode:
    """제약식 LHS를 나타내는 표현식 트리 노드.

    각 노드는 partial assignment 상태에서 (lo, hi) 구간을 반환할 수 있다.
    lo = 현재 도메인 하에서 최소값, hi = 현재 도메인 하에서 최대값.
    """
    pass

class ConstNode(ExprNode):
    """정수 상수."""
    __slots__ = ('val',)
    def __init__(self, val):
        self.val = int(val)

    def interval(self, domains):
        return (self.val, self.val)

    def evaluate(self, assignment):
        return self.val

    def variables(self):
        return set()

    def __repr__(self):
        return str(self.val)

class VarNode(ExprNode):
    """심볼릭 변수 (sp_X_Y)."""
    __slots__ = ('name',)
    def __init__(self, name):
        self.name = name

    def interval(self, domains):
        dom = domains.get(self.name)
        if dom is None:
            return (1, 1)
        if isinstance(dom, int):
            return (dom, dom)
        # dom은 정렬된 리스트
        return (dom[0], dom[-1])

    def evaluate(self, assignment):
        return assignment.get(self.name, 1)

    def variables(self):
        return {self.name}

    def __repr__(self):
        return self.name

class MulNode(ExprNode):
    """곱셈: left * right."""
    __slots__ = ('left', 'right')
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def interval(self, domains):
        a_lo, a_hi = self.left.interval(domains)
        b_lo, b_hi = self.right.interval(domains)
        # 모든 인수는 ≥1 이므로 단조증가
        return (a_lo * b_lo, a_hi * b_hi)

    def evaluate(self, assignment):
        return self.left.evaluate(assignment) * self.right.evaluate(assignment)

    def variables(self):
        return self.left.variables() | self.right.variables()

    def __repr__(self):
        return f"({self.left}*{self.right})"

class AddNode(ExprNode):
    """덧셈: left + right."""
    __slots__ = ('left', 'right')
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def interval(self, domains):
        a_lo, a_hi = self.left.interval(domains)
        b_lo, b_hi = self.right.interval(domains)
        return (a_lo + b_lo, a_hi + b_hi)

    def evaluate(self, assignment):
        return self.left.evaluate(assignment) + self.right.evaluate(assignment)

    def variables(self):
        return self.left.variables() | self.right.variables()

    def __repr__(self):
        return f"({self.left}+{self.right})"

class SubNode(ExprNode):
    """뺄셈: left - right."""
    __slots__ = ('left', 'right')
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def interval(self, domains):
        a_lo, a_hi = self.left.interval(domains)
        b_lo, b_hi = self.right.interval(domains)
        return (a_lo - b_hi, a_hi - b_lo)

    def evaluate(self, assignment):
        return self.left.evaluate(assignment) - self.right.evaluate(assignment)

    def variables(self):
        return self.left.variables() | self.right.variables()

    def __repr__(self):
        return f"({self.left}-{self.right})"

class MinNode(ExprNode):
    """min(left, right)."""
    __slots__ = ('left', 'right')
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def interval(self, domains):
        a_lo, a_hi = self.left.interval(domains)
        b_lo, b_hi = self.right.interval(domains)
        return (builtins_min(a_lo, b_lo), builtins_min(a_hi, b_hi))

    def evaluate(self, assignment):
        return builtins_min(self.left.evaluate(assignment), self.right.evaluate(assignment))

    def variables(self):
        return self.left.variables() | self.right.variables()

    def __repr__(self):
        return f"min({self.left},{self.right})"

class CeilDivNode(ExprNode):
    """ceil(left / right)."""
    __slots__ = ('left', 'right')
    def __init__(self, left, right):
        self.left = left
        self.right = right

    def interval(self, domains):
        a_lo, a_hi = self.left.interval(domains)
        b_lo, b_hi = self.right.interval(domains)
        # ceil(a/b): lo when a small and b large, hi when a large and b small
        b_lo = max(b_lo, 1)  # 0 나누기 방지
        b_hi = max(b_hi, 1)
        lo = (a_lo + b_hi - 1) // b_hi
        hi = (a_hi + b_lo - 1) // b_lo
        return (max(lo, 1), max(hi, 1))

    def evaluate(self, assignment):
        a = self.left.evaluate(assignment)
        b = self.right.evaluate(assignment)
        b = max(b, 1)
        return (a + b - 1) // b

    def variables(self):
        return self.left.variables() | self.right.variables()

    def __repr__(self):
        return f"ceil({self.left}/{self.right})"

class ScaleMulNode(ExprNode):
    """child * constant (dtype_bytes 등 곱). 최적화된 단축형."""
    __slots__ = ('child', 'scale')
    def __init__(self, child, scale):
        self.child = child
        self.scale = scale

    def interval(self, domains):
        lo, hi = self.child.interval(domains)
        return (lo * self.scale, hi * self.scale)

    def evaluate(self, assignment):
        return self.child.evaluate(assignment) * self.scale

    def variables(self):
        return self.child.variables()

    def __repr__(self):
        return f"({self.child}*{self.scale})"

class SumNode(ExprNode):
    """여러 자식의 합. shared memory 총합에 사용."""
    __slots__ = ('children',)
    def __init__(self, children):
        self.children = list(children)

    def interval(self, domains):
        lo = sum(c.interval(domains)[0] for c in self.children)
        hi = sum(c.interval(domains)[1] for c in self.children)
        return (lo, hi)

    def evaluate(self, assignment):
        return sum(c.evaluate(assignment) for c in self.children)

    def variables(self):
        v = set()
        for c in self.children:
            v |= c.variables()
        return v

    def __repr__(self):
        return " + ".join(str(c) for c in self.children)


def _parse_expr_tree(sym_expr_str):
    """SymExpr 문자열을 ExprNode 트리로 파싱.

    지원하는 문법:
      - 정수 리터럴
      - sp_X_Y 변수
      - a*b (곱)
      - (expr)
      - min(a,b)
      - ceil(a/(b))
      - a - b, a + b (스텐실 패턴)

    Returns: ExprNode
    """
    s = sym_expr_str.strip()
    return _parse_add_sub(s, 0)[0]

def _parse_add_sub(s, pos):
    """+ / - 파싱 (가장 낮은 우선순위)."""
    left, pos = _parse_mul(s, pos)
    while pos < len(s):
        if s[pos] == '+':
            right, pos = _parse_mul(s, pos + 1)
            left = AddNode(left, right)
        elif s[pos] == '-' and pos > 0:
            right, pos = _parse_mul(s, pos + 1)
            left = SubNode(left, right)
        elif s[pos] in ' ':
            # " - " 또는 " + " 패턴 처리
            j = pos
            while j < len(s) and s[j] == ' ':
                j += 1
            if j < len(s) and s[j] == '+':
                j2 = j + 1
                while j2 < len(s) and s[j2] == ' ':
                    j2 += 1
                right, pos = _parse_mul(s, j2)
                left = AddNode(left, right)
            elif j < len(s) and s[j] == '-':
                j2 = j + 1
                while j2 < len(s) and s[j2] == ' ':
                    j2 += 1
                right, pos = _parse_mul(s, j2)
                left = SubNode(left, right)
            else:
                break
        else:
            break
    return left, pos

def _parse_mul(s, pos):
    """* 파싱."""
    left, pos = _parse_atom(s, pos)
    while pos < len(s) and s[pos] == '*':
        right, pos = _parse_atom(s, pos + 1)
        left = MulNode(left, right)
    return left, pos

def _parse_atom(s, pos):
    """원자: 정수, 변수, 괄호, min(...), ceil(...)."""
    # 공백 스킵
    while pos < len(s) and s[pos] == ' ':
        pos += 1

    if pos >= len(s):
        return ConstNode(1), pos

    # 괄호
    if s[pos] == '(':
        inner, pos = _parse_add_sub(s, pos + 1)
        if pos < len(s) and s[pos] == ')':
            pos += 1
        return inner, pos

    # min(...)
    if s[pos:pos+4] == 'min(':
        pos += 4
        a, pos = _parse_add_sub(s, pos)
        if pos < len(s) and s[pos] == ',':
            pos += 1
        b, pos = _parse_add_sub(s, pos)
        if pos < len(s) and s[pos] == ')':
            pos += 1
        return MinNode(a, b), pos

    # ceil(.../(...)...)
    if s[pos:pos+5] == 'ceil(':
        pos += 5
        a, pos = _parse_add_sub(s, pos)
        if pos < len(s) and s[pos] == '/':
            pos += 1
        # slash 뒤에 ( 가 올 수 있음
        if pos < len(s) and s[pos] == '(':
            b, pos = _parse_add_sub(s, pos + 1)
            if pos < len(s) and s[pos] == ')':
                pos += 1
        else:
            b, pos = _parse_atom(s, pos)
        if pos < len(s) and s[pos] == ')':
            pos += 1
        return CeilDivNode(a, b), pos

    # math.ceil(...) — eval_sym_extent가 생성하는 형태
    if s[pos:pos+10] == 'math.ceil(':
        pos += 10
        a, pos = _parse_add_sub(s, pos)
        if pos < len(s) and s[pos] == '/':
            pos += 1
        if pos < len(s) and s[pos] == '(':
            b, pos = _parse_add_sub(s, pos + 1)
            if pos < len(s) and s[pos] == ')':
                pos += 1
        else:
            b, pos = _parse_atom(s, pos)
        if pos < len(s) and s[pos] == ')':
            pos += 1
        return CeilDivNode(a, b), pos

    # 정수 또는 변수
    start = pos
    if s[pos].isdigit():
        while pos < len(s) and s[pos].isdigit():
            pos += 1
        return ConstNode(int(s[start:pos])), pos

    # 변수: sp_X_Y 등 (알파벳/밑줄/숫자)
    if s[pos].isalpha() or s[pos] == '_':
        while pos < len(s) and (s[pos].isalnum() or s[pos] == '_'):
            pos += 1
        name = s[start:pos]
        return VarNode(name), pos

    # 알 수 없는 문자 — 스킵
    return ConstNode(1), pos + 1



In [90]:

# ─────────────────────────────────────────────────────────────
#  ScheduleGenerator: HW 제약 기반 스케줄 파라미터 생성
# ─────────────────────────────────────────────────────────────
class ScheduleGenerator:
    """
    HW_PARAM 기반 제약식을 구축하고, 제약을 만족하는 파라미터를 생성하는 생성기.

    제약 목록:
      1) max_vectorize_bytes:  각 vectorize extent × dtype_bytes ≤ max_vector_bytes
      2) max_shared_memory:    block 당 shared memory 총합 ≤ max_shared_memory_per_block
      3) max_threads:          각 thread extent ≤ max_threads_per_block
      4) min_thread_extent:    spatial product > warp_size*2 이면 thread extent ≥ warp_size
      5) max_vthread:          각 vthread extent ≤ max_vthread_extent
      6) max_innermost_split:  각 SplitStep의 마지막 factor ≤ max_innermost_split_factor

    생성 알고리즘:
      1) 초기화: 각 변수의 약수 도메인 구성 + max_innermost_split 반영
      2) 제약 전처리: 표현식 트리 파싱 + interval arithmetic
      3) 변수 순서 결정: shared → thread → 나머지
      4) 순차 할당: LHS_min > RHS 이면 후보 제거
      5) 상태 업데이트: 관련 제약식만 갱신
    """

    DEFAULT_HW_PARAM = {
        'max_vector_bytes': 16,
        'max_shared_memory_per_block': 49152,
        'max_threads_per_block': 1024,
        'max_vthread_extent': 8,
        'warp_size': 32,
        'max_innermost_split_factor': 64,
    }

    # 제약 종류 키 목록 (전체)
    ALL_CONSTRAINT_KINDS = (
        'vectorize', 'shared_memory', 'max_threads',
        'min_thread', 'vthread', 'innermost_split',
    )

    def __init__(self, sym_state, hw_param=None, enabled_constraints=None):
        """
        Args:
            sym_state: SymbolicState 인스턴스 (apply_steps 완료 상태)
            hw_param: dict — HW 제약 파라미터. None이면 DEFAULT_HW_PARAM 사용.
            enabled_constraints: set[str] | None — 활성화할 제약 종류.
                None이면 전체 활성화. 예: {'vectorize', 'max_threads'}
                사용 가능 키: 'vectorize', 'shared_memory', 'max_threads',
                             'min_thread', 'vthread', 'innermost_split'
        """
        self.s = sym_state
        self.hw = dict(self.DEFAULT_HW_PARAM)
        if hw_param is not None:
            self.hw.update(hw_param)
        self.pm = SymParamManager(sym_state)

        # 활성 제약 종류
        if enabled_constraints is None:
            self._enabled = set(self.ALL_CONSTRAINT_KINDS)
        else:
            unknown = set(enabled_constraints) - set(self.ALL_CONSTRAINT_KINDS)
            if unknown:
                raise ValueError(f"Unknown constraint kinds: {unknown}")
            self._enabled = set(enabled_constraints)

        # ─── 전처리: 제약식 빌드 + 파싱 ───
        self._constraints = []   # list of parsed constraint dicts
        self._var_constraints = {}  # {var_name: [constraint_index, ...]}
        self._var_order = []     # 변수 할당 순서

        self._preprocess()

    # ═══════════════════════════════════════════════════════════
    # 1) 제약식 빌드 (기존 API 유지)
    # ═══════════════════════════════════════════════════════════

    # ─── 1) Vectorize constraint ───
    def build_vectorize_constraints(self):
        """각 vectorize iter에 대해 (sym_extent_expr, dtype_bytes, limit) 튜플 목록 반환.
        제약: eval(sym_extent) * dtype_bytes ≤ max_vector_bytes
        """
        limit = self.hw['max_vector_bytes']
        constraints = []
        for sid, iid, ext in self.s.get_vectorize_extents():
            dtype_bytes = self.s.stages[sid].dtype_bytes
            constraints.append({
                'stage_id': sid,
                'iter_id': iid,
                'sym_extent': ext,
                'dtype_bytes': dtype_bytes,
                'limit': limit,
                'desc': f"vectorize s{sid}.i{iid} ({self.s.stages[sid].op_name}): "
                        f"extent*{dtype_bytes} ≤ {limit}",
            })
        return constraints

    def check_vectorize(self, sym_map=None):
        """vectorize 제약 위반 목록 반환. 빈 리스트 = 통과."""
        if sym_map is None:
            sym_map = self.s.sym_map
        violations = []
        for c in self.build_vectorize_constraints():
            val = eval_sym_extent(c['sym_extent'], sym_map)
            if isinstance(val, int) and val * c['dtype_bytes'] > c['limit']:
                violations.append(f"{c['desc']}: actual={val}*{c['dtype_bytes']}={val*c['dtype_bytes']}")
        return violations

    # ─── 2) Shared memory constraint ───
    def build_shared_memory_constraints(self):
        """block 당 shared memory 총합 제약.
        각 .shared stage의 extent × dtype_bytes 의 합 ≤ limit.
        Returns: dict with sym_exprs list, dtype_bytes list, limit.
        """
        limit = self.hw['max_shared_memory_per_block']
        items = []
        for sid, op_name, ext in self.s.get_shared_memory_extents():
            dtype_bytes = self.s.stages[sid].dtype_bytes
            items.append({
                'stage_id': sid,
                'op_name': op_name,
                'sym_extent': ext,
                'dtype_bytes': dtype_bytes,
            })
        return {
            'items': items,
            'limit': limit,
            'desc': f"shared_memory: sum(extent*dtype_bytes) ≤ {limit}",
        }

    def check_shared_memory(self, sym_map=None):
        """shared memory 제약 위반 여부. 위반 시 문자열 리스트 반환."""
        if sym_map is None:
            sym_map = self.s.sym_map
        c = self.build_shared_memory_constraints()
        total = 0
        parts = []
        for item in c['items']:
            val = eval_sym_extent(item['sym_extent'], sym_map)
            if isinstance(val, int):
                bytes_used = val * item['dtype_bytes']
                total += bytes_used
                parts.append(f"{item['op_name']}={val}*{item['dtype_bytes']}={bytes_used}")
        if total > c['limit']:
            return [f"{c['desc']}: actual={total} ({' + '.join(parts)})"]
        return []

    # ─── 3) Max threads per block constraint ───
    def build_max_threads_constraints(self):
        """각 thread extent ≤ max_threads_per_block 제약 목록."""
        limit = self.hw['max_threads_per_block']
        constraints = []
        for sid, iid, ext in self.s.get_thread_extents():
            constraints.append({
                'stage_id': sid,
                'iter_id': iid,
                'sym_extent': ext,
                'limit': limit,
                'desc': f"max_threads s{sid}.i{iid} ({self.s.stages[sid].op_name}): "
                        f"extent ≤ {limit}",
            })
        return constraints

    def check_max_threads(self, sym_map=None):
        """max_threads 제약 위반 목록."""
        if sym_map is None:
            sym_map = self.s.sym_map
        violations = []
        for c in self.build_max_threads_constraints():
            val = eval_sym_extent(c['sym_extent'], sym_map)
            if isinstance(val, int) and val > c['limit']:
                violations.append(f"{c['desc']}: actual={val}")
        return violations

    # ─── 4) Min thread extent constraint ───
    def _find_orig_op(self, op_name):
        """sym_state stage의 op_name으로 compute_dag.ops에서 원본 op을 찾는다.
        .shared / .local 접미사가 있으면 제거하고 원본 이름으로 매칭."""
        base_name = op_name
        for suffix in ('.shared', '.local'):
            if base_name.endswith(suffix):
                base_name = base_name[:-len(suffix)]
                break
        for op in self.s.compute_dag.ops:
            if op.name == base_name:
                return op
        return None

    def build_min_thread_constraints(self):
        """spatial product > warp_size*2 인 stage의 thread extent ≥ warp_size 제약 목록.
        C++ InitThreadBind의 check_min_thread_extent 로직과 동일.
        
        C++ 원본: pop->root_iter_vars() (= axis + reduce_axis) 의 extent 곱이
        warp_size * 2 이하이면 min_thread check를 비활성화.
        .shared stage는 cooperative fetching으로 별도 처리되므로 제외."""
        warp_size = self.hw['warp_size']
        constraints = []
        for sid, iid, ext in self.s.get_thread_extents():
            stage_name = self.s.stages[sid].op_name
            # .shared stage는 cooperative fetching이므로 min_thread 체크 안 함
            if stage_name.endswith('.shared'):
                continue
            # compute_dag.ops에서 원본 op을 찾아 root_iter_vars extent 곱 계산
            orig_op = self._find_orig_op(stage_name)
            if orig_op is not None and hasattr(orig_op, 'axis'):
                total_space = 1
                # C++: pop->root_iter_vars() = axis + reduce_axis
                for ax in orig_op.axis:
                    if ax.dom is not None:
                        total_space *= int(ax.dom.extent)
                for ax in orig_op.reduce_axis:
                    if ax.dom is not None:
                        total_space *= int(ax.dom.extent)
                if total_space > warp_size * 2:
                    constraints.append({
                        'stage_id': sid,
                        'iter_id': iid,
                        'sym_extent': ext,
                        'min_val': warp_size,
                        'total_space': total_space,
                        'desc': f"min_thread s{sid}.i{iid} ({stage_name}): "
                                f"extent ≥ {warp_size} (space={total_space})",
                    })
        return constraints

    def check_min_thread(self, sym_map=None):
        """min_thread 제약 위반 목록."""
        if sym_map is None:
            sym_map = self.s.sym_map
        violations = []
        for c in self.build_min_thread_constraints():
            val = eval_sym_extent(c['sym_extent'], sym_map)
            if isinstance(val, int) and val < c['min_val']:
                violations.append(f"{c['desc']}: actual={val}")
        return violations

    # ─── 5) Max vthread extent constraint ───
    def build_vthread_constraints(self):
        """각 vthread extent ≤ max_vthread_extent 제약 목록."""
        limit = self.hw['max_vthread_extent']
        constraints = []
        for sid, iid, ext in self.s.get_vthread_extents():
            constraints.append({
                'stage_id': sid,
                'iter_id': iid,
                'sym_extent': ext,
                'limit': limit,
                'desc': f"max_vthread s{sid}.i{iid} ({self.s.stages[sid].op_name}): "
                        f"extent ≤ {limit}",
            })
        return constraints

    def check_vthread(self, sym_map=None):
        """vthread 제약 위반 목록."""
        if sym_map is None:
            sym_map = self.s.sym_map
        violations = []
        for c in self.build_vthread_constraints():
            val = eval_sym_extent(c['sym_extent'], sym_map)
            if isinstance(val, int) and val > c['limit']:
                violations.append(f"{c['desc']}: actual={val}")
        return violations

    # ─── 6) Max innermost split factor constraint ───
    def build_innermost_split_constraints(self):
        """각 SplitStep의 마지막 factor ≤ max_innermost_split_factor 제약 목록.
        마지막 factor = sp_{step_idx}_{last_length_idx}."""
        limit = self.hw['max_innermost_split_factor']
        sp_groups = self.pm._build_sp_groups()
        constraints = []
        for step_idx, names in sorted(sp_groups.items()):
            # 마지막 length의 sym_name
            last_name = names[-1]
            constraints.append({
                'sym_name': last_name,
                'step_idx': step_idx,
                'limit': limit,
                'desc': f"max_innermost_split {last_name}: value ≤ {limit}",
            })
        return constraints

    def check_innermost_split(self, sym_map=None):
        """innermost split factor 제약 위반 목록."""
        if sym_map is None:
            sym_map = self.s.sym_map
        violations = []
        for c in self.build_innermost_split_constraints():
            val = sym_map.get(c['sym_name'])
            if val is not None and isinstance(val, int) and val > c['limit']:
                violations.append(f"{c['desc']}: actual={val}")
        return violations

    # ─── 통합 체크 ───
    def check_all(self, sym_map=None):
        """모든 제약 위반을 한 번에 검사. 위반 목록 반환 (빈 리스트 = 모두 통과)."""
        violations = []
        if 'vectorize' in self._enabled:
            violations.extend(self.check_vectorize(sym_map))
        if 'shared_memory' in self._enabled:
            violations.extend(self.check_shared_memory(sym_map))
        if 'max_threads' in self._enabled:
            violations.extend(self.check_max_threads(sym_map))
        if 'min_thread' in self._enabled:
            violations.extend(self.check_min_thread(sym_map))
        if 'vthread' in self._enabled:
            violations.extend(self.check_vthread(sym_map))
        if 'innermost_split' in self._enabled:
            violations.extend(self.check_innermost_split(sym_map))
        return violations

    # ═══════════════════════════════════════════════════════════
    # 2) 전처리: 초기 도메인 + 제약 파싱 + 변수 순서
    # ═══════════════════════════════════════════════════════════

    def _preprocess(self):
        """생성기 초기화 시 1회 실행.

        1) 초기 도메인: 약수 후보 + max_innermost_split 반영
        2) 제약 전처리: 표현식 트리 파싱, 변수 집합, RHS 저장
        3) 변수 순서: shared → thread → 나머지 (비선형/등장횟수 기준)
        """
        import re

        sp_groups = self.pm._build_sp_groups()
        sp_extents = self.pm._build_sp_extents(sp_groups)

        # ──── 1) 초기 도메인 구성 ────
        innermost_limit = self.hw['max_innermost_split_factor']
        innermost_names = set()
        if 'innermost_split' in self._enabled:
            for step_idx, names in sp_groups.items():
                innermost_names.add(names[-1])

        self._sp_groups = sp_groups
        self._sp_extents = sp_extents
        self._ur_names = [n for n in self.s.sym_map if n.startswith("ur_")]
        self._all_sp_names = []  # 모든 SP 이름 (step 순서)
        for step_idx in sorted(sp_groups.keys()):
            self._all_sp_names.extend(sp_groups[step_idx])

        # 각 SP 변수의 초기 도메인 = 약수 리스트
        # 단, 도메인 계산은 그룹 내 선행 변수의 값이 결정된 뒤에야 가능하므로,
        # 여기서는 extent만 기록하고, 도메인은 randomize_params 시 동적 계산
        # 다만 innermost 제약은 그룹 내 마지막이므로 정적 상한으로 기록
        self._innermost_names = innermost_names

        # ──── 2) 제약식 파싱 ────
        self._constraints = []
        self._var_constraints = {}  # {var_name: [constraint indices]}

        def _add_constraint(expr_tree, rhs, kind, desc, is_upper=True):
            """제약식 등록. is_upper=True: LHS ≤ RHS, False: LHS ≥ RHS."""
            idx = len(self._constraints)
            vars_in = expr_tree.variables()
            has_nonlinear = self._has_nonlinear(expr_tree)
            self._constraints.append({
                'tree': expr_tree,
                'rhs': rhs,
                'vars': vars_in,
                'kind': kind,
                'desc': desc,
                'is_upper': is_upper,
                'has_nonlinear': has_nonlinear,
            })
            for v in vars_in:
                self._var_constraints.setdefault(v, []).append(idx)

        # (a) vectorize: extent * dtype_bytes ≤ max_vector_bytes
        if 'vectorize' in self._enabled:
            for c in self.build_vectorize_constraints():
                tree = _parse_expr_tree(str(c['sym_extent']))
                if c['dtype_bytes'] != 1:
                    tree = ScaleMulNode(tree, c['dtype_bytes'])
                _add_constraint(tree, c['limit'], 'vectorize', c['desc'], is_upper=True)

        # (b) shared memory: sum(extent_i * dtype_bytes_i) ≤ limit
        if 'shared_memory' in self._enabled:
            sm = self.build_shared_memory_constraints()
            if sm['items']:
                children = []
                for item in sm['items']:
                    tree = _parse_expr_tree(str(item['sym_extent']))
                    if item['dtype_bytes'] != 1:
                        tree = ScaleMulNode(tree, item['dtype_bytes'])
                    children.append(tree)
                sum_tree = SumNode(children) if len(children) > 1 else children[0]
                _add_constraint(sum_tree, sm['limit'], 'shared_memory', sm['desc'], is_upper=True)

        # (c) max_threads: extent ≤ max_threads_per_block
        if 'max_threads' in self._enabled:
            for c in self.build_max_threads_constraints():
                tree = _parse_expr_tree(str(c['sym_extent']))
                _add_constraint(tree, c['limit'], 'max_threads', c['desc'], is_upper=True)

        # (d) min_thread: extent ≥ warp_size
        if 'min_thread' in self._enabled:
            for c in self.build_min_thread_constraints():
                tree = _parse_expr_tree(str(c['sym_extent']))
                _add_constraint(tree, c['min_val'], 'min_thread', c['desc'], is_upper=False)

        # (e) vthread: extent ≤ max_vthread_extent
        if 'vthread' in self._enabled:
            for c in self.build_vthread_constraints():
                tree = _parse_expr_tree(str(c['sym_extent']))
                _add_constraint(tree, c['limit'], 'vthread', c['desc'], is_upper=True)

        # (f) innermost split: sp_X_last ≤ max_innermost_split_factor
        # 이미 초기 도메인에서 반영하므로 별도 제약으로 등록 안 함
        # (초기 도메인 상한 컷으로 충분)

        # ──── 3) 변수 할당 순서 결정 ────
        self._compute_var_order()

    @staticmethod
    def _has_nonlinear(node):
        """표현식 트리에 비선형 연산(min, ceil)이 포함되어 있는지 확인."""
        if isinstance(node, (MinNode, CeilDivNode)):
            return True
        if isinstance(node, (MulNode, AddNode, SubNode)):
            return ScheduleGenerator._has_nonlinear(node.left) or ScheduleGenerator._has_nonlinear(node.right)
        if isinstance(node, (ScaleMulNode,)):
            return ScheduleGenerator._has_nonlinear(node.child)
        if isinstance(node, SumNode):
            return any(ScheduleGenerator._has_nonlinear(c) for c in node.children)
        return False

    def _compute_var_order(self):
        """변수 할당 순서를 결정한다.

        우선순위:
          1) shared_memory 제약에만 등장하는 변수 먼저
          2) thread 관련 (max_threads, min_thread) 제약 변수
          3) 나머지 (vectorize, vthread 등)

        동일 우선순위 내에서:
          1) 비선형 연산이 없는 제약식의 변수 우선
          2) 현재 + 다른 미처리 제약식에서 등장 횟수 합이 큰 변수 우선

        단, SP 그룹 내 변수는 length_idx 순서를 유지해야 하므로
        그룹 단위로 우선순위를 결정한 뒤, 그룹 내부는 원래 순서 유지.
        """
        # 각 변수가 참여하는 제약 종류 분류
        shared_vars = set()
        thread_vars = set()
        other_vars = set()

        for ci, c in enumerate(self._constraints):
            kind = c['kind']
            vs = c['vars']
            if kind == 'shared_memory':
                shared_vars.update(vs)
            elif kind in ('max_threads', 'min_thread'):
                thread_vars.update(vs)
            else:
                other_vars.update(vs)

        # 각 SP 그룹에 우선순위 부여
        # 그룹 우선순위: 0 = shared, 1 = thread, 2 = other
        # 세부 우선순위: (비선형 여부, -등장횟수 합)

        # 변수별 전체 등장 횟수
        var_freq = {}
        for v in self._all_sp_names:
            var_freq[v] = len(self._var_constraints.get(v, []))

        group_priority = {}
        for step_idx, group in self._sp_groups.items():
            group_set = set(group)

            # 그룹이 어떤 제약 종류에 속하는지 판별
            in_shared = bool(group_set & shared_vars)
            in_thread = bool(group_set & thread_vars)

            if in_shared:
                cat = 0
            elif in_thread:
                cat = 1
            else:
                cat = 2

            # 그룹 내 변수가 참여하는 제약들의 비선형 여부 최소값
            min_nonlinear = True
            total_freq = 0
            for v in group:
                total_freq += var_freq.get(v, 0)
                for ci in self._var_constraints.get(v, []):
                    if not self._constraints[ci]['has_nonlinear']:
                        min_nonlinear = False

            group_priority[step_idx] = (cat, min_nonlinear, -total_freq)

        # 정렬 (extent가 없는 그룹은 마지막)
        sorted_steps = sorted(
            self._sp_groups.keys(),
            key=lambda si: group_priority.get(si, (3, True, 0))
        )

        self._var_order = []
        for step_idx in sorted_steps:
            self._var_order.extend(self._sp_groups[step_idx])

    # ═══════════════════════════════════════════════════════════
    # 3) 제약 만족 파라미터 생성
    # ═══════════════════════════════════════════════════════════

    def randomize_params(self, rng=None, max_retries=1):
        """모든 HW 제약을 만족하는 파라미터를 약수 조건 + HW 제약 하에 랜덤 생성.

        알고리즘:
          1) 모든 SP를 초기값(1)으로 설정, domains = {var: [lo, hi] 또는 int}
          2) 변수 순서대로 하나씩 할당:
             a) 약수 조건으로 후보 리스트 구성
             b) innermost 제약 반영 (정적 상한)
             c) 도메인 상한으로 후보 사전 필터링
             d) 관련 제약식에 대해 interval + concrete eval 검사로 후보 필터링
             e) 남은 후보 중 랜덤 선택
          3) 상태 업데이트: 선택된 변수 확정 + 관련 제약식의 다른 변수 도메인 갱신

        Args:
            rng: random.Random 인스턴스 또는 None
            max_retries: 사후 rejection 최대 재시도 횟수

        Returns:
            dict: 설정된 {sym_name: value} 매핑

        Raises:
            RuntimeError: max_retries 초과 시
        """
        import random as _random
        if rng is None:
            rng = _random.Random()

        innermost_limit = self.hw['max_innermost_split_factor']

        violations = None
        for attempt in range(max_retries):
            result = {}
            # 모든 SP를 1로 초기화
            for name in self._all_sp_names:
                self.s.sym_map[name] = 1

            # 도메인 상태: {var_name: int (확정값) | [lo, hi] (미확정 유효 범위)}
            domains = {}
            for name in self._all_sp_names:
                parts = name.split("_")
                step_idx = int(parts[1])
                ext = self._sp_extents.get(step_idx)
                if ext is not None:
                    domains[name] = [1, ext]  # 미확정: [최소, 최대]
                else:
                    domains[name] = 1  # extent 없으면 1 고정

            # 그룹별 remaining extent 추적
            group_remaining = {}
            for step_idx, ext in self._sp_extents.items():
                group_remaining[step_idx] = ext

            ok = True
            for name in self._var_order:
                parts = name.split("_")
                step_idx = int(parts[1])
                length_idx = int(parts[2])

                extent = self._sp_extents.get(step_idx)
                if extent is None:
                    # extent 없는 SP: 원본 값 유지
                    result[name] = self.s.sym_map[name]
                    domains[name] = self.s.sym_map[name]
                    continue

                remaining = group_remaining.get(step_idx, extent)
                candidates = self.pm._divisors(remaining)

                # 1) innermost 제약: 그룹의 마지막 변수 ≤ limit
                if name in self._innermost_names:
                    candidates = [c for c in candidates if c <= innermost_limit]

                # 2) 도메인 상한/하한으로 사전 필터링
                dom = domains.get(name)
                if isinstance(dom, list):
                    if dom[1] < candidates[-1]:
                        candidates = [c for c in candidates if c <= dom[1]]
                    if dom[0] > candidates[0]:
                        candidates = [c for c in candidates if c >= dom[0]]

                # 3) 관련 제약식에 대해 interval + concrete eval 검사
                constraint_indices = self._var_constraints.get(name, [])
                if constraint_indices:
                    candidates = self._filter_by_constraints(
                        name, candidates, constraint_indices, domains)

                if not candidates:
                    candidates = [1]

                chosen = rng.choice(candidates)
                self.s.sym_map[name] = chosen
                result[name] = chosen
                domains[name] = chosen

                # 그룹 remaining 갱신
                group_remaining[step_idx] = (remaining + chosen - 1) // chosen

                # 4) 관련 제약식의 다른 미확정 변수 도메인 갱신
                if constraint_indices:
                    self._propagate_domain(name, domains)

            # unroll
            for name in self._ur_names:
                chosen = rng.choice(self.pm.UNROLL_CANDIDATES)
                self.s.sym_map[name] = chosen
                result[name] = chosen

            # 사후 검증
            violations = self.check_all()
            if not violations:
                return result

        raise RuntimeError(
            f"Failed to find valid params after {max_retries} retries. "
            f"Last violations: {violations}")

    def _propagate_domain(self, assigned_name, domains):
        """변수 확정 후, 관련 제약식의 다른 미확정 변수 도메인을 축소.

        확정된 변수가 참여하는 각 제약식에 대해:
          - upper 제약 (LHS ≤ RHS): 나머지 미확정 변수 각각에 대해
            concrete eval로 그 변수의 유효 상한을 이진 검색으로 축소
          - lower 제약 (LHS ≥ RHS): 나머지 미확정 변수 각각에 대해
            concrete eval로 그 변수의 유효 하한을 이진 검색으로 축소

        이는 constraint propagation (arc consistency)의 경량 버전이다.
        """
        constraint_indices = self._var_constraints.get(assigned_name, [])
        sym_map = self.s.sym_map

        for ci in constraint_indices:
            c = self._constraints[ci]
            tree = c['tree']
            rhs = c['rhs']
            is_upper = c['is_upper']

            # 이 제약식의 미확정 변수들
            for other_var in c['vars']:
                if other_var == assigned_name:
                    continue
                dom = domains.get(other_var)
                if not isinstance(dom, list):
                    continue  # 이미 확정됨

                cur_lo, cur_hi = dom

                if is_upper:
                    # LHS ≤ RHS: other_var의 상한 축소
                    # other_var = cur_hi 일 때 LHS_concrete 계산
                    old_val = sym_map.get(other_var, 1)
                    sym_map[other_var] = cur_hi
                    lhs_at_hi = tree.evaluate(sym_map)
                    if lhs_at_hi <= rhs:
                        sym_map[other_var] = old_val
                        continue  # 현재 상한으로 충분

                    # cur_lo 에서도 위반이면 상한을 lo로 축소
                    sym_map[other_var] = cur_lo
                    lhs_at_lo = tree.evaluate(sym_map)
                    if lhs_at_lo > rhs:
                        # cur_lo 조차 위반 — 도메인을 [1, 1]로 축소
                        sym_map[other_var] = old_val
                        dom[1] = cur_lo
                        continue

                    # 이진 검색: 유효한 최대값 찾기
                    lo, hi = cur_lo, cur_hi
                    while lo < hi:
                        mid = (lo + hi + 1) // 2
                        sym_map[other_var] = mid
                        lhs_val = tree.evaluate(sym_map)
                        if lhs_val <= rhs:
                            lo = mid
                        else:
                            hi = mid - 1

                    sym_map[other_var] = old_val
                    if lo < cur_hi:
                        dom[1] = lo  # 상한 축소

                else:
                    # LHS ≥ RHS: other_var의 하한 축소
                    old_val = sym_map.get(other_var, 1)
                    sym_map[other_var] = cur_lo
                    lhs_at_lo = tree.evaluate(sym_map)
                    if lhs_at_lo >= rhs:
                        sym_map[other_var] = old_val
                        continue  # 현재 하한으로 충분

                    sym_map[other_var] = cur_hi
                    lhs_at_hi = tree.evaluate(sym_map)
                    if lhs_at_hi < rhs:
                        # other_var를 최대로 올려도 LHS가 RHS에 못 미침.
                        # 이는 다른 미확정 변수의 값에 따라 달라질 수 있으므로
                        # 도메인을 축소하지 않고 건너뜀.
                        sym_map[other_var] = old_val
                        continue

                    # 이진 검색: 유효한 최소값 찾기
                    lo, hi = cur_lo, cur_hi
                    while lo < hi:
                        mid = (lo + hi) // 2
                        sym_map[other_var] = mid
                        lhs_val = tree.evaluate(sym_map)
                        if lhs_val >= rhs:
                            hi = mid
                        else:
                            lo = mid + 1

                    sym_map[other_var] = old_val
                    if lo > cur_lo:
                        dom[0] = lo  # 하한 축소

    def _filter_by_constraints(self, var_name, candidates, constraint_indices, domains):
        """후보 리스트에서 제약을 위반하는 값을 제거.

        각 후보 v에 대해:
          1) domains[var_name] = v 로 가정
          2) 관련 제약식의 LHS interval을 계산
          3) upper 제약: LHS_min > RHS → 제거
             lower 제약: LHS_max < RHS → 제거

        이진 검색 최적화: upper 제약에서 후보가 정렬 상태이고
        LHS가 var_name에 대해 단조증가이면, 최대 유효 인덱스를 이진 검색으로 탐색.

        Args:
            var_name: 현재 결정 중인 SP 이름
            candidates: 정렬된 후보 리스트
            constraint_indices: 관련 제약 인덱스 리스트
            domains: 현재 도메인 상태

        Returns:
            list: 유효한 후보 리스트
        """
        if not candidates:
            return candidates

        # 도메인 스냅샷 (interval 계산용): 확정된 변수는 int, 미확정은 [lo, hi]
        # domains는 이미 _propagate_domain에 의해 축소된 [lo, hi]를 가지고 있음
        interval_domains = {}
        for v, d in domains.items():
            interval_domains[v] = d  # int → (d, d), [lo, hi] → (lo, hi)

        # 후보 필터링: 이진 검색 + 선형 검사 혼합
        # upper 제약 (LHS ≤ RHS): 이진 검색 가능 (후보 증가 → LHS_min 증가)
        # lower 제약 (LHS ≥ RHS): 이진 검색 가능 (후보 증가 → LHS_max 증가)
        upper_constraints = []
        lower_constraints = []
        for ci in constraint_indices:
            c = self._constraints[ci]
            if c['is_upper']:
                upper_constraints.append(c)
            else:
                lower_constraints.append(c)

        # ─── upper 제약: 이진 검색으로 최대 유효 인덱스 ───
        max_valid_idx = len(candidates) - 1
        for c in upper_constraints:
            idx = self._bisect_upper(var_name, candidates, c, interval_domains)
            max_valid_idx = builtins_min(max_valid_idx, idx)

        # ─── upper 제약: concrete eval 이진 검색 (interval 보완) ───
        # interval arithmetic는 sub-expression 별로 독립 min/max를 합성하므로
        # product-of-min 같은 표현식에서 지나치게 느슨해질 수 있다.
        # concrete eval: 확정 변수는 실제 값, 미확정 변수는 1 → 더 타이트한 하한.
        sym_map_snap = dict(self.s.sym_map)  # 현재 sym_map 스냅샷
        for c in upper_constraints:
            idx = self._bisect_upper_concrete(
                var_name, candidates, c, sym_map_snap, max_valid_idx)
            max_valid_idx = builtins_min(max_valid_idx, idx)

        # ─── lower 제약: 이진 검색으로 최소 유효 인덱스 ───
        min_valid_idx = 0
        for c in lower_constraints:
            idx = self._bisect_lower(var_name, candidates, c, interval_domains)
            min_valid_idx = max(min_valid_idx, idx)

        if min_valid_idx > max_valid_idx:
            return [candidates[0]]

        return candidates[min_valid_idx:max_valid_idx + 1]

    def _bisect_upper(self, var_name, candidates, constraint, interval_domains):
        """upper 제약 (LHS ≤ RHS)에 대해 유효한 최대 후보 인덱스를 이진 검색.

        후보가 커질수록 LHS_min이 증가하는 단조성을 이용.
        LHS_min > RHS 이면 해당 후보 이상은 모두 무효.
        """
        tree = constraint['tree']
        rhs = constraint['rhs']

        lo, hi = 0, len(candidates) - 1

        # 빠른 경로: 최대 후보 검사
        test_dom = dict(interval_domains)
        test_dom[var_name] = candidates[hi]
        lhs_min, _ = tree.interval(test_dom)
        if lhs_min <= rhs:
            return hi  # 모두 유효

        # 빠른 경로: 최소 후보 검사
        test_dom[var_name] = candidates[lo]
        lhs_min, _ = tree.interval(test_dom)
        if lhs_min > rhs:
            return -1  # 모두 무효 (candidates[0] 이라도 반환하게 됨)

        # 이진 검색
        while lo < hi:
            mid = (lo + hi + 1) // 2
            test_dom[var_name] = candidates[mid]
            lhs_min, _ = tree.interval(test_dom)
            if lhs_min <= rhs:
                lo = mid
            else:
                hi = mid - 1

        return lo

    def _bisect_lower(self, var_name, candidates, constraint, interval_domains):
        """lower 제약 (LHS ≥ RHS)에 대해 유효한 최소 후보 인덱스를 이진 검색.

        후보가 커질수록 LHS_max가 증가하는 단조성을 이용.
        LHS_max < RHS 이면 해당 후보 이하는 모두 무효.
        """
        tree = constraint['tree']
        rhs = constraint['rhs']

        lo, hi = 0, len(candidates) - 1

        # 빠른 경로: 최소 후보 검사
        test_dom = dict(interval_domains)
        test_dom[var_name] = candidates[lo]
        _, lhs_max = tree.interval(test_dom)
        if lhs_max >= rhs:
            return lo  # 모두 유효

        # 빠른 경로: 최대 후보 검사
        test_dom[var_name] = candidates[hi]
        _, lhs_max = tree.interval(test_dom)
        if lhs_max < rhs:
            return hi + 1  # 모두 무효

        # 이진 검색
        while lo < hi:
            mid = (lo + hi) // 2
            test_dom[var_name] = candidates[mid]
            _, lhs_max = tree.interval(test_dom)
            if lhs_max >= rhs:
                hi = mid
            else:
                lo = mid + 1

        return lo

    def _bisect_upper_concrete(self, var_name, candidates, constraint,
                               sym_map_snap, current_max_idx):
        """upper 제약 (LHS ≤ RHS)에 대해 concrete eval 기반 이진 검색.

        interval arithmetic의 한계를 보완한다.
        확정 변수 → 실제 값, 미확정 변수 → 1 로 evaluate 하면
        product-of-min 등에서 interval보다 타이트한 하한을 얻을 수 있다.

        tree.evaluate(assignment)는 모든 인수 ≥ 1일 때
        var_name 값이 커질수록 단조 비감소이므로 이진 검색이 가능하다.
        """
        tree = constraint['tree']
        rhs = constraint['rhs']

        hi = builtins_min(current_max_idx, len(candidates) - 1)
        if hi < 0:
            return -1
        lo = 0

        # 빠른 경로: 최대 후보 검사
        sym_map_snap[var_name] = candidates[hi]
        lhs_val = tree.evaluate(sym_map_snap)
        if lhs_val <= rhs:
            return hi  # 모두 유효

        # 빠른 경로: 최소 후보 검사
        sym_map_snap[var_name] = candidates[lo]
        lhs_val = tree.evaluate(sym_map_snap)
        if lhs_val > rhs:
            sym_map_snap[var_name] = 1  # 복원
            return -1  # 모두 무효

        # 이진 검색
        while lo < hi:
            mid = (lo + hi + 1) // 2
            sym_map_snap[var_name] = candidates[mid]
            lhs_val = tree.evaluate(sym_map_snap)
            if lhs_val <= rhs:
                lo = mid
            else:
                hi = mid - 1

        sym_map_snap[var_name] = 1  # 복원 (아직 확정 전)
        return lo

    # ═══════════════════════════════════════════════════════════
    # 4) DFS 전수 열거
    # ═══════════════════════════════════════════════════════════

    def enumerate_all_params(self, max_results=100_000):
        """DFS로 모든 제약-만족 SP 파라미터 조합을 열거한다.

        unroll 변수는 SP 열거 후 카르테시안 곱으로 결합한다.

        알고리즘:
          randomize_params와 동일한 순서(self._var_order)로 변수를 결정하되,
          rng.choice 대신 유효 후보 전체에 대해 재귀 분기한다.

        Args:
            max_results: 최대 결과 수 (안전 장치)

        Returns:
            list[dict]: 각 원소는 {sp_X_Y: int, ..., ur_X: int, ...} 매핑
        """
        from itertools import product as itertools_product

        innermost_limit = self.hw['max_innermost_split_factor']
        sp_results = []

        # ── DFS 본체 ──
        def _dfs(var_idx, result, domains, group_remaining):
            if len(sp_results) >= max_results:
                return

            if var_idx == len(self._var_order):
                # 모든 SP 확정 → 사후 검증
                for name, val in result.items():
                    self.s.sym_map[name] = val
                violations = self.check_all({**result})
                if not violations:
                    sp_results.append(dict(result))
                return

            name = self._var_order[var_idx]
            parts = name.split("_")
            step_idx = int(parts[1])

            extent = self._sp_extents.get(step_idx)
            if extent is None:
                result[name] = self.s.sym_map.get(name, 1)
                domains[name] = result[name]
                _dfs(var_idx + 1, result, domains, group_remaining)
                del result[name]
                del domains[name]
                return

            remaining = group_remaining.get(step_idx, extent)
            candidates = self.pm._divisors(remaining)

            # 1) innermost 제약
            if name in self._innermost_names:
                candidates = [c for c in candidates if c <= innermost_limit]

            # 2) 도메인 상한/하한 필터
            dom = domains.get(name)
            if isinstance(dom, list):
                if dom[1] < candidates[-1]:
                    candidates = [c for c in candidates if c <= dom[1]]
                if dom[0] > candidates[0]:
                    candidates = [c for c in candidates if c >= dom[0]]

            # 3) 제약 필터
            constraint_indices = self._var_constraints.get(name, [])
            if constraint_indices:
                candidates = self._filter_by_constraints(
                    name, candidates, constraint_indices, domains)

            if not candidates:
                return  # 이 분기는 해 없음 — 백트래킹

            old_sym = self.s.sym_map.get(name, 1)
            old_remaining = group_remaining.get(step_idx, extent)

            for chosen in candidates:
                if len(sp_results) >= max_results:
                    return

                self.s.sym_map[name] = chosen
                result[name] = chosen

                saved_domains = {}
                for k, v in domains.items():
                    if isinstance(v, list):
                        saved_domains[k] = list(v)
                domains[name] = chosen

                group_remaining[step_idx] = (remaining + chosen - 1) // chosen

                # 도메인 전파
                if constraint_indices:
                    self._propagate_domain(name, domains)

                _dfs(var_idx + 1, result, domains, group_remaining)

                # 복원
                del result[name]
                self.s.sym_map[name] = old_sym
                group_remaining[step_idx] = old_remaining
                for k, saved_v in saved_domains.items():
                    domains[k] = saved_v
                domains.pop(name, None)

        # ── 초기 상태 ──
        for name in self._all_sp_names:
            self.s.sym_map[name] = 1

        domains = {}
        for name in self._all_sp_names:
            parts = name.split("_")
            step_idx = int(parts[1])
            ext = self._sp_extents.get(step_idx)
            if ext is not None:
                domains[name] = [1, ext]
            else:
                domains[name] = 1

        group_remaining = {}
        for step_idx, ext in self._sp_extents.items():
            group_remaining[step_idx] = ext

        _dfs(0, {}, domains, group_remaining)

        # ── unroll 카르테시안 곱 ──
        if not sp_results:
            return []

        ur_names = sorted(self._ur_names)
        if not ur_names:
            return sp_results

        ur_combos = list(itertools_product(
            *[self.pm.UNROLL_CANDIDATES for _ in ur_names]))

        all_results = []
        for sp in sp_results:
            for ur_vals in ur_combos:
                if len(all_results) >= max_results:
                    return all_results
                combined = dict(sp)
                for i, ur_name in enumerate(ur_names):
                    combined[ur_name] = ur_vals[i]
                all_results.append(combined)

        return all_results



In [24]:
# ═══════════════════════════════════════════════════════════════
#  HW 파라미터 설정 (CUDA sm_86)
# ═══════════════════════════════════════════════════════════════
HW_PARAM = {
    'max_vector_bytes': 16,
    'max_shared_memory_per_block': 49152,
    'max_threads_per_block': 1024,
    'max_vthread_extent': 8,
    'warp_size': 32,
    'max_innermost_split_factor': 64,
}

# grouped 요약 출력
n_tasks = len(grouped)
n_sketches = sum(len(v) for v in grouped.values())
n_records = sum(len(recs) for v in grouped.values() for recs in v.values())
print(f"Tasks: {n_tasks}, Sketches: {n_sketches}, Total records: {n_records}")
print(f"HW params: {HW_PARAM}")

Tasks: 29, Sketches: 33, Total records: 54219
HW params: {'max_vector_bytes': 16, 'max_shared_memory_per_block': 49152, 'max_threads_per_block': 1024, 'max_vthread_extent': 8, 'warp_size': 32, 'max_innermost_split_factor': 64}


In [25]:
# ═══════════════════════════════════════════════════════════════
#  TVM API를 통한 스케줄 유효성 검증 유틸리티
# ═══════════════════════════════════════════════════════════════
import json, copy, random
from tvm import tir
from tvm.auto_scheduler.measure_record import load_record_from_string, dump_record_to_string

# ─── ScheduleToModule + GPU passes ───
_s2m = tvm.get_global_func("driver.schedule_to_module")
GPU_PASSES = tvm.transform.Sequential(
    [
        tir.transform.InjectPrefetch(),
        tir.transform.StorageFlatten(64, False),
        tir.transform.NarrowDataType(32),
        tir.transform.Simplify(),
        tir.transform.VectorizeLoop(True),
        tir.transform.InjectVirtualThread(),
        tir.transform.StorageRewrite(),
        tir.transform.Simplify(),
    ]
)

GPU_VERIFY_CONSTRAINTS = {
    "max_shared_memory_per_block": 49152,
    "max_local_memory_per_block": 2**31 - 1,
    "max_threads_per_block": 1024,
    "max_thread_x": 1024,
    "max_thread_y": 1024,
    "max_thread_z": 64,
    "max_vthread": 8,
    "max_vector_bytes": 16,
}


def lower_with_gpu_passes(task, state):
    """State → TE schedule → IRModule → GPU pass pipeline 적용."""
    sch, tensors = task.compute_dag.apply_steps_from_state(state)
    mod = _s2m(sch, tensors, "main", {})
    return GPU_PASSES(mod)


def verify_gpu_module(mod, constraints=None):
    """Lowered IRModule에 대해 tir.analysis.verify_gpu_code로 GPU 제약 확인."""
    if constraints is None:
        constraints = GPU_VERIFY_CONSTRAINTS
    verify_fn = tvm.get_global_func("tir.analysis.verify_gpu_code")
    for _, f in mod.functions.items():
        if isinstance(f, tvm.tir.PrimFunc):
            if not verify_fn(f, constraints):
                return False
    return True


def params_to_state(task, base_inp, base_res, params):
    """ScheduleGenerator가 생성한 params를 적용한 새 auto_scheduler State를 반환."""
    record_str = dump_record_to_string(base_inp, base_res)
    record = json.loads(record_str)
    steps = record["i"][1][1]

    for name, val in params.items():
        if name.startswith("sp_"):
            parts = name.split("_")
            step_idx = int(parts[1])
            length_idx = int(parts[2])
            s = steps[step_idx]
            if s[0] == "SP" and length_idx < len(s[4]):
                s[4][length_idx] = int(val)
        elif name.startswith("ur_"):
            parts = name.split("_")
            step_idx = int(parts[1])
            s = steps[step_idx]
            if s[0] == "PR":
                s[3] = f"auto_unroll_max_step${int(val)}"

    patched_str = json.dumps(record)
    new_inp, _ = load_record_from_string(patched_str)
    return new_inp.state


print("✅ TVM API 검증 유틸리티 로드 완료")

✅ TVM API 검증 유틸리티 로드 완료


In [92]:
# ═══════════════════════════════════════════════════════════════
#  전체 TVM API 검증:
#    grouped (JSON record) → sketch별 첫 state로 sym_state 생성
#    → ScheduleGenerator → randomize_params → params_to_state
#    → lower_with_gpu_passes → verify_gpu_module
#    + 중복 파라미터 사후 제거 & 카운트
# ═══════════════════════════════════════════════════════════════
import random, time

N_TRIALS = 500
SEED = 42

total_ok = 0
total_dup = 0
total_gen_fail = 0
total_apply_fail = 0
total_lower_fail = 0
total_verify_fail = 0
fail_details_v = []

t0 = time.time()

def _params_fingerprint(params):
    """params dict → 정렬된 (key, value) 튜플로 해싱 가능한 fingerprint 생성."""
    return tuple(sorted(params.items()))

for task_idx, wkey in enumerate(grouped.keys()):
    task = tasks[task_idx]

    for sketch_idx, sketch_fp in enumerate(grouped[wkey].keys()):
        records = grouped[wkey][sketch_fp]

        # 첫 번째 record 사용
        base_inp, base_res = records[0]
        base_state = base_inp.state

        # sym_state 생성
        sym_state = build_symbolic_state(task.compute_dag, base_state)

        task_ok = 0
        task_dup = 0
        task_fail_gen = 0
        task_fail_apply = 0
        task_fail_lower = 0
        task_fail_verify = 0
        seen_fps = set()  # 이 sketch 내 중복 감지용

        new_states = []

        for trial in range(N_TRIALS):
            rng = random.Random(SEED + trial)
            gen = ScheduleGenerator(sym_state)

            # 1) 파라미터 생성
            try:
                params = gen.randomize_params(rng=rng, max_retries=1)
            except RuntimeError as e:
                task_fail_gen += 1
                if len(fail_details_v) < 30:
                    fail_details_v.append(f"T{task_idx} S{sketch_idx} trial{trial} [gen]: {e}")
                continue

            # 2) 중복 검사
            fp = _params_fingerprint(params)
            if fp in seen_fps:
                task_dup += 1
                continue
            seen_fps.add(fp)

            # 3) params → State
            try:
                new_state = params_to_state(task, base_inp, base_res, params)
                new_states.append(new_state)
            except Exception as e:
                task_fail_apply += 1
                if len(fail_details_v) < 30:
                    fail_details_v.append(f"T{task_idx} S{sketch_idx} trial{trial} [state]: {e}")
                continue

            # 4) apply_steps → TE schedule
            try:
                sch, tensors = task.compute_dag.apply_steps_from_state(new_state)
            except Exception as e:
                task_fail_apply += 1
                if len(fail_details_v) < 30:
                    fail_details_v.append(f"T{task_idx} S{sketch_idx} trial{trial} [apply]: {e}")
                continue

            # 5) lower + GPU passes
            try:
                mod = lower_with_gpu_passes(task, new_state)
            except Exception as e:
                task_fail_lower += 1
                if len(fail_details_v) < 30:
                    fail_details_v.append(f"T{task_idx} S{sketch_idx} trial{trial} [lower]: {e}")
                continue

            # 6) verify_gpu_module
            try:
                ok = verify_gpu_module(mod)
            except Exception as e:
                task_fail_verify += 1
                if len(fail_details_v) < 30:
                    fail_details_v.append(f"T{task_idx} S{sketch_idx} trial{trial} [verify_exc]: {e}")
                continue

            if ok:
                task_ok += 1
            else:
                task_fail_verify += 1
                if len(fail_details_v) < 30:
                    fail_details_v.append(
                        f"T{task_idx} S{sketch_idx} trial{trial} [verify_FAIL]: GPU constraint violated")

        total_ok += task_ok
        total_dup += task_dup
        total_gen_fail += task_fail_gen
        total_apply_fail += task_fail_apply
        total_lower_fail += task_fail_lower
        total_verify_fail += task_fail_verify

        task_unique = task_ok + task_fail_apply + task_fail_lower + task_fail_verify
        task_total = task_unique + task_dup + task_fail_gen
        task_failures = task_fail_apply + task_fail_lower + task_fail_verify
        status = "✅" if task_failures == 0 else "❌"
        extra_parts = []
        if task_dup: extra_parts.append(f"dup={task_dup}")
        if task_fail_gen: extra_parts.append(f"gen={task_fail_gen}")
        if task_fail_apply: extra_parts.append(f"apply={task_fail_apply}")
        if task_fail_lower: extra_parts.append(f"lower={task_fail_lower}")
        if task_fail_verify: extra_parts.append(f"verify={task_fail_verify}")
        extra = f"  ({', '.join(extra_parts)})" if extra_parts else ""
        print(f"  T{task_idx:2d} S{sketch_idx}: {task_ok}/{task_unique} unique, "
              f"{N_TRIALS} trials {status}{extra}  [{task.desc}]")
    break

elapsed = time.time() - t0
total_unique = total_ok + total_apply_fail + total_lower_fail + total_verify_fail
total_all = total_unique + total_dup + total_gen_fail
total_failures = total_apply_fail + total_lower_fail + total_verify_fail
print(f"\n{'='*70}")
print(f"Trials: {total_all}  |  Unique: {total_unique}  |  Duplicates: {total_dup}  |  "
      f"({elapsed:.1f}s)")
print(f"  ✅ Pass:         {total_ok}")
print(f"  🔁 Duplicate:    {total_dup}")
print(f"  ❌ Gen fail:     {total_gen_fail}")
print(f"  ❌ Apply fail:   {total_apply_fail}")
print(f"  ❌ Lower fail:   {total_lower_fail}")
print(f"  ❌ Verify fail:  {total_verify_fail}")

if fail_details_v:
    print(f"\n실패 상세 (최대 30건):")
    for d in fail_details_v:
        print(f"  {d}")

if total_failures == 0 and total_gen_fail == 0:
    print(f"\n✅✅✅ ALL {total_unique} UNIQUE PASSED — "
          f"(+{total_dup} duplicates skipped) ✅✅✅")

  T 0 S0: 500/500 unique, 500 trials ✅  [vm_mod_fused_nn_batch_matmul_3]

Trials: 500  |  Unique: 500  |  Duplicates: 0  |  (9.7s)
  ✅ Pass:         500
  🔁 Duplicate:    0
  ❌ Gen fail:     0
  ❌ Apply fail:   0
  ❌ Lower fail:   0
  ❌ Verify fail:  0

✅✅✅ ALL 500 UNIQUE PASSED — (+0 duplicates skipped) ✅✅✅


In [85]:
print(str(tasks[0].compute_dag.infer_bound_from_state(records[4][0].state)))

Placeholder: p0, p1
blockIdx.x b.0@i.0@j.0@ (0,64)
  vthread b.1@i.1@j.1@ (0,4)
    threadIdx.x b.2@i.2@j.2@ (0,8)
      T_batch_matmul_NN.local auto_unroll: 1024
      for b_c.0 (0,1)
        for i_c.0 (0,1)
          for j_c.0 (0,1)
            for b_c.1 (0,1)
              for i_c.1 (0,1)
                for j_c.1 (0,1)
                  for b_c.2 (0,1)
                    for i_c.2 (0,1)
                      for j_c.2 (0,1)
                        for k.0 (0,64)
                          for ax0@ax1@ax2@.0.0 (0,128)
                            threadIdx.x ax0@ax1@ax2@.0.1 (0,8)
                              vectorize ax0@ax1@ax2@.1 (0,1)
                                p1.shared = ...
                          for ax0@ax1@ax2@.0.0 (0,64)
                            threadIdx.x ax0@ax1@ax2@.0.1 (0,8)
                              vectorize ax0@ax1@ax2@.1 (0,1)
                                p0.shared = ...
                          for k.1 (0,8)
                            for b_c

In [95]:
for n_s in new_states:
    print(str(tasks[0].compute_dag.infer_bound_from_state(n_s)))

Placeholder: p0, p1
blockIdx.x b.0@i.0@j.0@ (0,4)
  vthread b.1@i.1@j.1@ (0,8)
    threadIdx.x b.2@i.2@j.2@ (0,1024)
      T_batch_matmul_NN.local auto_unroll: 1024
      for b_c.0 (0,1)
        for i_c.0 (0,1)
          for j_c.0 (0,1)
            for b_c.1 (0,1)
              for i_c.1 (0,1)
                for j_c.1 (0,1)
                  for b_c.2 (0,1)
                    for i_c.2 (0,1)
                      for j_c.2 (0,1)
                        for k.0 (0,1024)
                          for ax0@ax1@ax2@.0.0 (0,1)
                            threadIdx.x ax0@ax1@ax2@.0.1 (0,1024)
                              vectorize ax0@ax1@ax2@.1 (0,1)
                                p1.shared = ...
                          for ax0@ax1@ax2@.0.0 (0,1)
                            threadIdx.x ax0@ax1@ax2@.0.1 (0,1024)
                              vectorize ax0@ax1@ax2@.1 (0,4)
                                p0.shared = ...
                          for k.1 (0,1)
                            

In [91]:
# ═══════════════════════════════════════════════════════════════
#  enabled_constraints 기능 테스트
# ═══════════════════════════════════════════════════════════════
print("=== enabled_constraints 기능 테스트 ===\n")

# 테스트용 sym_state 생성 (첫 번째 task, 첫 번째 sketch)
_test_wkey = list(grouped.keys())[0]
_test_sketches = grouped[_test_wkey]
_test_sk_fp = list(_test_sketches.keys())[0]
_test_recs = _test_sketches[_test_sk_fp]
_test_inp, _test_res = _test_recs[0]
_test_ss = build_symbolic_state(tasks[0].compute_dag, _test_inp.state)

# 1) 모든 제약 활성 (기본값)
gen_all = ScheduleGenerator(_test_ss, HW_PARAM)
print(f"1) 모든 제약 활성 (기본값):")
print(f"   enabled: {sorted(gen_all._enabled)}")
print(f"   constraints: {len(gen_all._constraints)}개")
print(f"   innermost_names: {len(gen_all._innermost_names)}개")

# 2) 모든 제약 비활성
gen_none = ScheduleGenerator(_test_ss, HW_PARAM, enabled_constraints=set())
print(f"\n2) 모든 제약 비활성:")
print(f"   enabled: {sorted(gen_none._enabled)}")
print(f"   constraints: {len(gen_none._constraints)}개")
print(f"   innermost_names: {len(gen_none._innermost_names)}개")

# 3) shared_memory + max_threads만 활성
gen_partial = ScheduleGenerator(_test_ss, HW_PARAM,
    enabled_constraints={'shared_memory', 'max_threads'})
print(f"\n3) shared_memory + max_threads만 활성:")
print(f"   enabled: {sorted(gen_partial._enabled)}")
print(f"   constraints: {len(gen_partial._constraints)}개")
kinds_found = set(c['kind'] for c in gen_partial._constraints)
print(f"   등록된 제약 종류: {sorted(kinds_found)}")
print(f"   innermost_names: {len(gen_partial._innermost_names)}개")

# 4) 잘못된 제약 이름 → ValueError
try:
    ScheduleGenerator(_test_ss, HW_PARAM, enabled_constraints={'invalid_name'})
    print("\n4) ValueError 미발생 ❌")
except ValueError as e:
    print(f"\n4) ValueError 정상 발생: {e} ✅")

# 5) check_all은 비활성 제약 스킵하는지 검증
params_all = gen_all.randomize_params()
v_all = gen_all.check_all()
v_none = gen_none.check_all(params_all)
print(f"\n5) check_all 비활성 제약 스킵 검증:")
print(f"   전체 활성 violations: {len(v_all)}")
print(f"   전체 비활성 violations (같은 params): {len(v_none)}")
assert len(v_none) == 0, "비활성 제약에서 violation이 나오면 안 됨"
print(f"   ✅ 비활성 제약 정상 스킵됨")

# 6) 개별 제약 하나씩 활성화하여 constraint 종류 확인
print(f"\n6) 개별 제약 활성화 테스트:")
for kind in ScheduleGenerator.ALL_CONSTRAINT_KINDS:
    gen_k = ScheduleGenerator(_test_ss, HW_PARAM, enabled_constraints={kind})
    n_c = len(gen_k._constraints)
    kinds_in = set(c['kind'] for c in gen_k._constraints)
    innermost = len(gen_k._innermost_names)
    extra = ""
    if kind == 'innermost_split':
        extra = f", innermost_names={innermost}"
    print(f"   {kind:20s}: {n_c}개 제약, 종류={sorted(kinds_in)}{extra}")
    # 등록된 제약 종류가 해당 kind만 포함해야 함 (innermost_split은 constraints 미사용)
    if kind != 'innermost_split':
        assert kinds_in <= {kind}, f"기대하지 않은 제약 종류: {kinds_in}"
    else:
        assert n_c == 0, f"innermost_split은 _constraints에 등록되면 안 됨"
        assert innermost > 0, "innermost_split 활성인데 innermost_names가 비어 있으면 안 됨"

print("\n=== 모든 enabled_constraints 테스트 통과 ✅ ===")

=== enabled_constraints 기능 테스트 ===

1) 모든 제약 활성 (기본값):
   enabled: ['innermost_split', 'max_threads', 'min_thread', 'shared_memory', 'vectorize', 'vthread']
   constraints: 8개
   innermost_names: 6개

2) 모든 제약 비활성:
   enabled: []
   constraints: 0개
   innermost_names: 0개

3) shared_memory + max_threads만 활성:
   enabled: ['max_threads', 'shared_memory']
   constraints: 4개
   등록된 제약 종류: ['max_threads', 'shared_memory']
   innermost_names: 0개

4) ValueError 정상 발생: Unknown constraint kinds: {'invalid_name'} ✅

5) check_all 비활성 제약 스킵 검증:
   전체 활성 violations: 0
   전체 비활성 violations (같은 params): 0
   ✅ 비활성 제약 정상 스킵됨

6) 개별 제약 활성화 테스트:
   vectorize           : 2개 제약, 종류=['vectorize']
   shared_memory       : 1개 제약, 종류=['shared_memory']
   max_threads         : 3개 제약, 종류=['max_threads']
   min_thread          : 1개 제약, 종류=['min_thread']
   vthread             : 1개 제약, 종류=['vthread']
   innermost_split     : 0개 제약, 종류=[], innermost_names=6

=== 모든 enabled_constraints 테스트 통과 ✅ ===


In [77]:
# ── 진단: trial=0에서 실제로 어떤 SP 값이 선택되고 왜 실패하는지 ──
task = tasks[0]
dag = task.compute_dag
wkey = list(grouped.keys())[0]
sketch_fp_key = list(grouped[wkey].keys())[0]
recs = grouped[wkey][sketch_fp_key]
base_inp, base_res = recs[0]
base_state = base_inp.state

sym_state = build_symbolic_state(dag, base_state)

import random

rng = random.Random(42)
gen = ScheduleGenerator(sym_state)

# randomize_params를 수동으로 실행하면서 각 단계를 트레이스
innermost_limit = gen.hw['max_innermost_split_factor']

for name in gen._all_sp_names:
    gen.s.sym_map[name] = 1

domains = {}
for name in gen._all_sp_names:
    parts = name.split("_")
    step_idx = int(parts[1])
    ext = gen._sp_extents.get(step_idx)
    if ext is not None:
        domains[name] = [1, ext]
    else:
        domains[name] = 1

group_remaining = {}
for step_idx, ext in gen._sp_extents.items():
    group_remaining[step_idx] = ext

print("=== 단계별 SP 선택 트레이스 ===\n")
result = {}
for name in gen._var_order:
    parts = name.split("_")
    step_idx = int(parts[1])
    length_idx = int(parts[2])

    extent = gen._sp_extents.get(step_idx)
    if extent is None:
        result[name] = gen.s.sym_map[name]
        domains[name] = gen.s.sym_map[name]
        continue

    remaining = group_remaining.get(step_idx, extent)
    candidates = gen.pm._divisors(remaining)

    if name in gen._innermost_names:
        candidates = [c for c in candidates if c <= innermost_limit]

    dom = domains.get(name)
    if isinstance(dom, list) and dom[1] < candidates[-1]:
        candidates = [c for c in candidates if c <= dom[1]]

    constraint_indices = gen._var_constraints.get(name, [])
    pre_filter_candidates = list(candidates)
    if constraint_indices:
        candidates = gen._filter_by_constraints(name, candidates, constraint_indices, domains)

    if not candidates:
        candidates = [1]

    chosen = rng.choice(candidates)
    gen.s.sym_map[name] = chosen
    result[name] = chosen
    domains[name] = chosen

    group_remaining[step_idx] = (remaining + chosen - 1) // chosen

    # 출력
    dom_after = {k: v for k, v in domains.items()
                 if isinstance(v, list) and k.startswith(f'sp_{step_idx}_')}
    print(f"{name:12s} = {chosen:4d}  (remaining={remaining:6d}, "
          f"pre_filter={pre_filter_candidates}, "
          f"post_filter={candidates[:10]}{'...' if len(candidates)>10 else ''})")

    if constraint_indices:
        gen._propagate_domain(name, domains)
        # propagation 후 변화된 도메인 출력
        changed = {}
        for k, v in domains.items():
            if isinstance(v, list):
                changed[k] = v
        if changed:
            print(f"             propagated domains: {changed}")

print(f"\n=== 최종 sym_map ===")
print(f"  {result}")

# check_min_thread
violations = gen.check_min_thread(result)
print(f"\nmin_thread violations: {violations}")

# 실제 thread extent 계산
c6 = gen._constraints[6]
val = eval_sym_extent(c6['tree'], result)
print(f"thread extent = {val}")

=== 단계별 SP 선택 트레이스 ===

sp_1_0       =    1  (remaining=     1, pre_filter=[1], post_filter=[1])
             propagated domains: {'sp_1_1': [1, 1], 'sp_1_2': [1, 1], 'sp_1_3': [1, 1], 'sp_2_0': [1, 8], 'sp_2_1': [1, 64], 'sp_2_2': [1, 64], 'sp_2_3': [1, 64], 'sp_3_0': [1, 8], 'sp_3_1': [1, 512], 'sp_3_2': [1, 512], 'sp_3_3': [1, 512], 'sp_4_0': [1, 2048], 'sp_4_1': [1, 2048], 'sp_22_0': [1, 16], 'sp_27_0': [1, 32]}
sp_1_1       =    1  (remaining=     1, pre_filter=[1], post_filter=[1])
             propagated domains: {'sp_1_2': [1, 1], 'sp_1_3': [1, 1], 'sp_2_0': [1, 8], 'sp_2_1': [32, 64], 'sp_2_2': [1, 64], 'sp_2_3': [1, 64], 'sp_3_0': [1, 8], 'sp_3_1': [32, 512], 'sp_3_2': [1, 512], 'sp_3_3': [1, 512], 'sp_4_0': [1, 2048], 'sp_4_1': [1, 2048], 'sp_22_0': [1, 16], 'sp_27_0': [1, 32]}
sp_1_2       =    1  (remaining=     1, pre_filter=[1], post_filter=[1])
             propagated domains: {'sp_1_3': [1, 1], 'sp_2_0': [1, 8], 'sp_2_1': [32, 64], 'sp_2_2': [1, 64], 'sp_2_3': [1, 64],

In [31]:
# ═══════════════════════════════════════════════════════════════
#  DFS 전수 열거: 후보가 적은 sketch에 대해 모든 유효 스케줄 생성
#  + TVM API 검증 (lower + verify_gpu_module)
#
#  전략: 모든 sketch에서 DFS 열거를 시도하되,
#        열거 결과가 DFS_MAX_RESULTS를 초과하면 "공간이 큼"으로 판단하여 스킵.
#        작은 공간만 실제 TVM 검증까지 수행.
# ═══════════════════════════════════════════════════════════════
import time

DFS_MAX_RESULTS = 50_000   # 이 이하만 전수 검증
DFS_SKIP_LARGE = True      # True면 큰 공간은 열거만 스킵

t0 = time.time()

dfs_total_enumerated = 0
dfs_total_pass = 0
dfs_total_fail = 0
dfs_skipped_sketches = 0
dfs_fail_details = []

for task_idx, wkey in enumerate(grouped.keys()):
    task = tasks[task_idx]

    for sketch_idx, sketch_fp in enumerate(grouped[wkey].keys()):
        records = grouped[wkey][sketch_fp]
        base_inp, base_res = records[0]
        base_state = base_inp.state

        sym_state = build_symbolic_state(task.compute_dag, base_state)
        gen = ScheduleGenerator(sym_state)

        # DFS 전수 열거 시도
        all_params = gen.enumerate_all_params(max_results=DFS_MAX_RESULTS)
        n_enum = len(all_params)

        if n_enum >= DFS_MAX_RESULTS:
            # 탐색 공간이 너무 큼 → 스킵
            dfs_skipped_sketches += 1
            print(f"  T{task_idx:2d} S{sketch_idx}: ⏭  ≥{DFS_MAX_RESULTS} combos (skipped)"
                  f"  [{task.desc}]")
            continue

        # TVM API 검증
        sketch_ok = 0
        sketch_fail = 0

        for pi, params in enumerate(all_params):
            try:
                new_state = params_to_state(task, base_inp, base_res, params)
                mod = lower_with_gpu_passes(task, new_state)
                ok = verify_gpu_module(mod)
            except Exception as e:
                ok = False

            if ok:
                sketch_ok += 1
            else:
                sketch_fail += 1
                if len(dfs_fail_details) < 30:
                    dfs_fail_details.append(
                        f"T{task_idx} S{sketch_idx} param#{pi}: {params}")

        dfs_total_enumerated += n_enum
        dfs_total_pass += sketch_ok
        dfs_total_fail += sketch_fail

        status = "✅" if sketch_fail == 0 else "❌"
        print(f"  T{task_idx:2d} S{sketch_idx}: {status} {n_enum:,} combos, "
              f"{sketch_ok}/{n_enum} pass  [{task.desc}]")

elapsed = time.time() - t0

print(f"\n{'='*70}")
print(f"DFS 전수 열거 결과  ({elapsed:.1f}s)")
print(f"  Sketches 열거:    {33 - dfs_skipped_sketches}/{33}")
print(f"  Sketches 스킵:    {dfs_skipped_sketches} (≥{DFS_MAX_RESULTS} combos)")
print(f"  총 열거:          {dfs_total_enumerated:,}")
print(f"  ✅ Pass:          {dfs_total_pass:,}")
print(f"  ❌ Fail:          {dfs_total_fail}")

if dfs_fail_details:
    print(f"\n실패 상세 (최대 30건):")
    for d in dfs_fail_details:
        print(f"  {d}")

if dfs_total_fail == 0 and dfs_total_enumerated > 0:
    print(f"\n✅✅✅ ALL {dfs_total_enumerated:,} DFS-ENUMERATED PASSED ✅✅✅")

  T 0 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu]
  T 1 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu_1]
  T 2 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu_2]
  T 3 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_contrib_conv2d_winograd_without_weight_transform_add_nn_relu_3]
  T 4 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_conv2d]
  T 5 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_conv2d_1]
  T 6 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_conv2d_2]
  T 7 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_conv2d_3]
  T 8 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_conv2d_add]
  T 9 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_conv2d_add_1]
  T10 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_conv2d_add_2]
  T11 S0: ⏭  ≥50000 combos (skipped)  [vm_mod_fused_nn_conv2d_a